In [ ]:
# =====================================================
# BREAKOUT PRE-ANNOTATION PIPELINE 
# =====================================================
#
# Purpose:
#   Interpretable pre-annotation of breakout candidates in acoustic
#   borehole image logs 
#
# Output:
#   - QC panels
#   - binary masks
#   - overlay images
#   - automatic detection table
#   - bilateral-only visual QC table with manual + visual_match columns
#
# Notes:
#   - Layer and vertical-streak suppression are used for candidate detection.
#   - Final labeling uses the non-layer-suppressed cleaned mask.
#   - This is a pre-annotation / visual QC workflow, not final interpretation.

# Output:
#   One folder per preprocessing mode, each with QC panels and masks.

import os
import re
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.cluster import KMeans
from scipy.ndimage import gaussian_filter1d
from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.measure import label, regionprops

# =====================================================
# PATHS
# =====================================================

INPUT_FOLDER = r"your input folder here"
OUTPUT_ROOT = os.path.join(INPUT_FOLDER, "breakout_preannotator")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

VALID_EXTENSIONS = (".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".png")

# =====================================================
# PREPROCESSING MODE
# =====================================================

PREPROCESS_MODE = "bilateral_only"
PREPROCESS_MODES = [PREPROCESS_MODE]  # kept as a one-item list for compatibility

BILATERAL_D = 7
BILATERAL_SIGMA_COLOR = 22
BILATERAL_SIGMA_SPACE = 9

# =====================================================
# KMEANS
# =====================================================

N_CLUSTERS = 6
KMEANS_SAMPLE_PIXELS = 250000   # prevents crashes on huge images; set None for full image
KMEANS_RANDOM_STATE = 42

# =====================================================
# SEGMENTATION ABLATION
# =====================================================

SEGMENTATION_MODES = ["kmeans"]


# =====================================================
# MASK CLEANUP
# =====================================================

MIN_OBJECT_SIZE = 40
MAX_HOLE_SIZE = 60

# =====================================================
# LAYER SUPPRESSION
# =====================================================

MAX_ROW_DARK_FRACTION = 0.35
MIN_LAYER_RUN = 6
LAYER_DILATION_Y = 9

# Hard layer exclusion:
# if a dark pixel belongs to a laterally continuous layer row,
# it is removed from the final breakout annotation.
EXCLUDE_LAYER_PIXELS_FROM_FINAL_MASK = False

# Component-level layer veto:
# remove final components mostly supported by layer rows.
ENABLE_FINAL_LAYER_COMPONENT_VETO = False
MAX_FINAL_COMPONENT_LAYER_OVERLAP_FRACTION = 0.30
MAX_COMPONENT_LAYER_ROW_FRACTION = 0.40

ENABLE_WINDOW_LAYER_OVERLAP_VETO = False
MAX_WINDOW_LAYER_PIXEL_FRACTION = 0.35
MAX_WINDOW_LAYER_ROW_FRACTION = 0.45

MAX_COMPONENT_WIDTH_FRACTION = 0.35
MAX_HORIZONTAL_ASPECT = 3.5

# =====================================================
# ARTIFACT / VERTICAL-LINE SUPPRESSION
# =====================================================

ENABLE_VERTICAL_ARTIFACT_REJECTION = True
MIN_BREAKOUT_COMPONENT_WIDTH_PX = 5
MAX_THIN_COMPONENT_WIDTH_PX = 3
MIN_ARTIFACT_HEIGHT_FRACTION = 0.18
MAX_ARTIFACT_FILL_FRACTION = 0.35
EDGE_ARTIFACT_MARGIN_FRACTION = 0.06

ENABLE_PRE_AZIMUTH_VERTICAL_LINE_SUPPRESSION = True
VERTICAL_LINE_KERNEL_HEIGHT_FRACTION = 0.08
VERTICAL_LINE_KERNEL_MIN_HEIGHT_PX = 80
VERTICAL_LINE_KERNEL_WIDTH_PX = 3
VERTICAL_LINE_MAX_BBOX_WIDTH_PX = 20
VERTICAL_LINE_MIN_HEIGHT_FRACTION = 0.08
VERTICAL_LINE_SUBTRACT_DILATION_X = 1

# =====================================================
# AZIMUTH PAIR DETECTION
# =====================================================

HOTSPOT_PERCENTILE = 62
MIN_REGION_WIDTH_DEG = 10

MIN_AZIMUTH_SEPARATION = 90
MAX_AZIMUTH_SEPARATION = 240

MIN_WIDTH_RATIO = 0.12
MAX_WIDTH_RATIO = 8.0

MIN_WEAK_LOBE_STRENGTH = 0.00035
MIN_PAIR_DOMINANCE = 0.08

VERTICALITY_WEIGHT = 0.25
MAX_DIRECTION_DRIFT = 120

# =====================================================
# SINGLE BREAKOUT FALLBACK
# =====================================================
# Used only as a fallback when no valid pair survives.
# This accepts a single dominant azimuth lobe only near 180 degrees.
ENABLE_SINGLE_BREAKOUT_MODE = True
SINGLE_BREAKOUT_TARGET_AZIMUTH_DEG = 200
SINGLE_BREAKOUT_AZIMUTH_TOLERANCE_DEG = 65

# Single lobe must be strong and dominant in the 1D azimuth signal.
MIN_SINGLE_BREAKOUT_STRENGTH = 0.00035
MIN_SINGLE_BREAKOUT_DOMINANCE = 0.18
MIN_SINGLE_BREAKOUT_PEAK_BG_RATIO = 1.40
MIN_SINGLE_BREAKOUT_ISOLATION = 1.1
MAX_SINGLE_BREAKOUT_WIDTH_DEG = 150

# If True, the final blob filter will not remove the accepted 180-degree
# single breakout just because it is tall / narrow / line-like.
PRESERVE_SINGLE_BREAKOUT_VERTICAL_BLOB = True

# =====================================================
# CONTINUITY SINGLE RESCUE
# =====================================================
# Uses confirmed_direction as the last accepted breakout azimuth, whether
# that previous image was pair, single, or continuity-rescued.
# This is checked only when pair + classic single detection fail.
ENABLE_CONTINUITY_SINGLE_RESCUE = True
CONTINUITY_SINGLE_RESCUE_TOL_DEG = 65
MIN_CONTINUITY_SINGLE_RESCUE_STRENGTH = 0.00015
MIN_CONTINUITY_SINGLE_PEAK_BG_RATIO = 1.00
MAX_CONTINUITY_SINGLE_WIDTH_DEG = 190

# Pair quality gate:
# reject unilateral clutter by requiring both lobes to have comparable
# vertical support and non-extreme mask area balance.
ENABLE_PAIR_QUALITY_GATE = True
MIN_PAIR_AREA_BALANCE = 0.15
MIN_PAIR_VERTICAL_OVERLAP = 0.20
PAIR_QC_AZIMUTH_MARGIN_PX = 6

# =====================================================
# EXTRA PAIR FALSE-POSITIVE GATES
# =====================================================
# These reject patterns like:
#   - one broad strong lobe + one tiny spike
#   - edge spike paired with a broad lobe
#   - accepted pair plus a stronger third competing peak elsewhere
ENABLE_PAIR_BALANCE_GATES = True
MIN_PAIR_STRENGTH_BALANCE = 0.45
MIN_PAIR_ENERGY_BALANCE = 0.35
MIN_PAIR_WIDTH_BALANCE = 0.30

ENABLE_EDGE_SPIKE_PAIR_VETO = True
EDGE_SPIKE_AZIMUTH_MARGIN_DEG = 30
MAX_EDGE_SPIKE_WIDTH_DEG = 35

ENABLE_COMPETING_PEAK_VETO = True
MAX_COMPETING_PEAK_TO_WEAK_PEAK_RATIO = 1.25
MAX_COMPETING_ENERGY_TO_WEAK_ENERGY_RATIO = 0.90


# =====================================================
# AZIMUTH SIGNAL VALLEY / PAIR ISOLATION GATE
# =====================================================
# Goal:
#   keep true bimodal breakout patterns:
#       peak -- clean valley -- peak
#   reject cave/megapore/fracture-rich multimodal signals:
#       many competing peaks, weak pair isolation, noisy saddle.
#
# These gates operate on the 1D azimuth signal before final mask growth.

ENABLE_VALLEY_GATE = True

# Valley must be deep relative to the weaker accepted peak.
MIN_INTERPEAK_VALLEY_DEPTH = 0.35

# Valley should not be excessively noisy relative to weaker peak.
MAX_INTERPEAK_VALLEY_NOISE = 0.50

# Weakest accepted pair peak should be clearly above background.
MIN_PEAK_TO_BACKGROUND_RATIO = 1.20

# Pair peaks should be stronger than other competing peaks.
ENABLE_PAIR_ISOLATION_GATE = False
MIN_PAIR_ISOLATION = 0.75

# Broad plateau / cave veto:
# rejects one broad high-energy plateau paired with a nearby/narrow lobe.
ENABLE_BROAD_PLATEAU_VETO = True
BROAD_PLATEAU_MAX_SEPARATION = 140
BROAD_PLATEAU_MIN_WIDTH_DEG = 70

# Cave-rich multimodal signal veto:
# if many candidate pairs are plausible, the signal is likely cave/megapore clutter.
ENABLE_CAVE_RICH_MULTIPAIR_VETO = True
MAX_CAVE_RICH_VALID_PAIRS = 2
ENABLE_PAIR_CONTRAST_VETO = True

MIN_SIDE_CONTRAST = -1.0
MIN_MEAN_PAIR_CONTRAST = 4.0

# Cave-rich weak-evidence veto:
# rejects pairs that pass basic valley QC but have low total dominance,
# weak peak-to-background contrast, and poor isolation.
ENABLE_CAVE_RICH_WEAK_EVIDENCE_VETO = True
CAVE_RICH_MIN_PAIR_DOMINANCE = 0.50
CAVE_RICH_MIN_PEAK_BG_RATIO = 2.00
CAVE_RICH_MIN_PAIR_ISOLATION = 1.15



# Effective lobe strength balances sharp peaks and broad lobe energy.
# This helps keep asymmetric real breakouts where one side is broad/shouldered.
EFFECTIVE_PEAK_P90_WEIGHT = 0.60
EFFECTIVE_PEAK_MEAN_WEIGHT = 0.40

# =====================================================
# IMAGE REGIME PRESET TOGGLE
# =====================================================
# Use:
#   IMAGE_REGIME = "cave_poor"  -> normal / cleaner wells, preserve recall
#   IMAGE_REGIME = "cave_rich"  -> cave-megapore-rich wells, reduce false positives
#
# The preset is applied once before the main loop.

IMAGE_REGIME = "cave_rich"


def apply_image_regime_preset():

    global ENABLE_PAIR_ISOLATION_GATE
    global MIN_INTERPEAK_VALLEY_DEPTH
    global MAX_INTERPEAK_VALLEY_NOISE
    global MIN_PEAK_TO_BACKGROUND_RATIO
    global BROAD_PLATEAU_MAX_SEPARATION
    global BROAD_PLATEAU_MIN_WIDTH_DEG
    global MIN_PAIR_AREA_BALANCE
    global MIN_PAIR_VERTICAL_OVERLAP
    global WIDTH_EXPANSION_PX
    global ADAPTIVE_EXPANSION_MAX_PX
    global MAX_CAVE_RICH_VALID_PAIRS
    global ENABLE_SINGLE_BREAKOUT_MODE
    global ENABLE_CONTINUITY_SINGLE_RESCUE
    global ENABLE_PAIR_CONTRAST_VETO
    global ENABLE_FINAL_DARKEST_PIXEL_GATE
    global FINAL_DARK_PIXEL_PERCENTILE
    global MIN_FINAL_BLOB_AREA_PX

    if IMAGE_REGIME == "cave_poor":

        ENABLE_PAIR_ISOLATION_GATE = False
        ENABLE_SINGLE_BREAKOUT_MODE = True
        ENABLE_CONTINUITY_SINGLE_RESCUE = True

        MIN_INTERPEAK_VALLEY_DEPTH = 0.35
        MAX_INTERPEAK_VALLEY_NOISE = 0.80
        MIN_PEAK_TO_BACKGROUND_RATIO = 1.15

        BROAD_PLATEAU_MAX_SEPARATION = 110
        BROAD_PLATEAU_MIN_WIDTH_DEG = 100

        MIN_PAIR_AREA_BALANCE = 0.12
        MIN_PAIR_VERTICAL_OVERLAP = 0.15

        WIDTH_EXPANSION_PX = 8
        ADAPTIVE_EXPANSION_MAX_PX = 16

        # disabled in cave-poor
        MAX_CAVE_RICH_VALID_PAIRS = 999

    elif IMAGE_REGIME == "cave_rich":

        ENABLE_PAIR_ISOLATION_GATE = False
        ENABLE_SINGLE_BREAKOUT_MODE = False
        ENABLE_CONTINUITY_SINGLE_RESCUE = True
        ENABLE_PAIR_CONTRAST_VETO = False
        MIN_INTERPEAK_VALLEY_DEPTH = 0.45
        MAX_INTERPEAK_VALLEY_NOISE = 0.55
        MIN_PEAK_TO_BACKGROUND_RATIO = 1.30
        ENABLE_FINAL_DARKEST_PIXEL_GATE = True
        FINAL_DARK_PIXEL_PERCENTILE = 55
        MIN_FINAL_BLOB_AREA_PX = 180

        BROAD_PLATEAU_MAX_SEPARATION = 140
        BROAD_PLATEAU_MIN_WIDTH_DEG = 70

        MIN_PAIR_AREA_BALANCE = 0.05
        MIN_PAIR_VERTICAL_OVERLAP = 0.25

        WIDTH_EXPANSION_PX = 6
        ADAPTIVE_EXPANSION_MAX_PX = 12

        MAX_CAVE_RICH_VALID_PAIRS = 2

    else:

        raise ValueError(
            "IMAGE_REGIME must be 'cave_poor' or 'cave_rich'"
        )

    print("================================================")
    print(f"Applied image regime preset: {IMAGE_REGIME}")
    print(f"ENABLE_PAIR_ISOLATION_GATE = {ENABLE_PAIR_ISOLATION_GATE}")
    print(f"MIN_INTERPEAK_VALLEY_DEPTH = {MIN_INTERPEAK_VALLEY_DEPTH}")
    print(f"MAX_INTERPEAK_VALLEY_NOISE = {MAX_INTERPEAK_VALLEY_NOISE}")
    print(f"MIN_PEAK_TO_BACKGROUND_RATIO = {MIN_PEAK_TO_BACKGROUND_RATIO}")
    print(f"BROAD_PLATEAU_MAX_SEPARATION = {BROAD_PLATEAU_MAX_SEPARATION}")
    print(f"BROAD_PLATEAU_MIN_WIDTH_DEG = {BROAD_PLATEAU_MIN_WIDTH_DEG}")
    print(f"MIN_PAIR_AREA_BALANCE = {MIN_PAIR_AREA_BALANCE}")
    print(f"MIN_PAIR_VERTICAL_OVERLAP = {MIN_PAIR_VERTICAL_OVERLAP}")
    print(f"WIDTH_EXPANSION_PX = {WIDTH_EXPANSION_PX}")
    print(f"ADAPTIVE_EXPANSION_MAX_PX = {ADAPTIVE_EXPANSION_MAX_PX}")
    print(f"MAX_CAVE_RICH_VALID_PAIRS = {MAX_CAVE_RICH_VALID_PAIRS}")
    print("================================================")


# =====================================================
# VERTICAL WINDOW SETTINGS
# =====================================================

WINDOW_HEIGHT = 3000
WINDOW_STEP = WINDOW_HEIGHT // 2

LOCAL_BREAKOUT_THRESHOLD = 0.045
MIN_LOCAL_COMPONENT_AREA = 15
MIN_LOCAL_VERTICALITY = 0.1

ENABLE_MIRROR_ASSISTED_VALIDATION = True
WEAK_LOCAL_BREAKOUT_THRESHOLD = 0.015
WEAK_MIN_LOCAL_COMPONENT_AREA = 6
WEAK_MIN_LOCAL_VERTICALITY = 0.08

# =====================================================
# FINAL FILL / GROWTH
# =====================================================

ENABLE_ACCEPTED_PAIR_RESCUE = True
PAIR_RESCUE_MIN_COMPONENT_AREA = 4
PAIR_RESCUE_MIN_COLUMN_DENSITY = 0.002

ENABLE_ACCEPTED_WINDOW_DARK_FILL = True
DARK_FILL_PERCENTILE =40
DARK_FILL_MIN_OBJECT_SIZE = 10

DARK_FILL_VERTICAL_KERNEL = 11
DARK_FILL_HORIZONTAL_KERNEL = 3

DARK_FILL_REQUIRE_SEED_CONNECTION = True
DARK_FILL_SEED_DILATION_Y = 30
DARK_FILL_SEED_DILATION_X = 5
DARK_FILL_MIN_SEED_PIXELS_PER_COMPONENT = 10
DARK_FILL_MAX_VERTICAL_GAP = 25

ENABLE_GROW_OUTSIDE_VALIDATION_BOX = True
LABEL_GROW_AZIMUTH_MARGIN_PX = 22
LABEL_GROW_VERTICAL_MARGIN_PX = 80
LABEL_GROW_USE_FULL_HEIGHT = False

ENABLE_WIDTH_EXPANSION = True
WIDTH_EXPANSION_PX = 8
WIDTH_EXPANSION_INTENSITY_RELAX = 6

ENABLE_ADAPTIVE_ROW_EXPANSION = True
ADAPTIVE_EXPANSION_MAX_PX = 16
ADAPTIVE_EXPANSION_RELAX = 3
ADAPTIVE_EXPANSION_MIN_SEED_PIXELS_PER_ROW = 1

# =====================================================
# FINAL DARKEST-PIXEL GATE
# =====================================================

ENABLE_FINAL_DARKEST_PIXEL_GATE = True
FINAL_DARK_PIXEL_PERCENTILE = 45
FINAL_DARK_GATE_AZIMUTH_MARGIN_PX = 45
FINAL_GATE_SEED_KERNEL_X = 5
FINAL_GATE_SEED_KERNEL_Y = 15

# =====================================================
# DIAGNOSTICS ONLY
# =====================================================

ENABLE_POST_FILL_QUALITY_REPORT = True
AZIMUTH_CONTRAST_MARGIN_PX = 18
DIAG_MIN_MEAN_BREAKOUT_WIDTH_PX = 7.0

# =====================================================
# FINAL CONNECTED-BLOB / STREAK FILTERS
# =====================================================

ENABLE_FINAL_BLOB_SIZE_FILTER = True
MIN_FINAL_BLOB_AREA_PX = 180

MAX_THIN_VERTICAL_WIDTH_PX = 3
THIN_VERTICAL_MIN_ASPECT = 10

MAX_THIN_HORIZONTAL_HEIGHT_PX = 4
THIN_HORIZONTAL_MIN_ASPECT = 10

ENABLE_LINE_LIKE_VERTICAL_STREAK_VETO = True
MIN_STREAK_HEIGHT_PX = 120
MIN_STREAK_ASPECT_RATIO = 8.0
MAX_STREAK_MEDIAN_ROW_WIDTH_PX = 12.0
MAX_STREAK_P90_ROW_WIDTH_PX = 18.0
MAX_STREAK_ROW_WIDTH_STD_PX = 6.0

ENABLE_FINAL_COLUMN_STREAK_SUPPRESSION = False
FINAL_COLUMN_STREAK_MIN_RUN_FRACTION = 0.18
FINAL_COLUMN_STREAK_MAX_LOCAL_WIDTH_PX = 10
FINAL_COLUMN_STREAK_LOCAL_HALF_WIDTH = 4

# Cave-rich post-fill veto:
# rejects masks that fragment into too many components after fill/gating.
# This targets cave-rich overfill false positives.
ENABLE_CAVE_RICH_POST_FILL_FRAGMENT_VETO = True
MAX_CAVE_RICH_FINAL_BLOBS = 80

# =====================================================
# FILES — SORT SHALLOW → DEEP
# first number = depth
# =====================================================

def extract_depth_key(filename):

    name = os.path.splitext(filename)[0]

    # first number in filename = depth
    match = re.search(r"\d+", name)

    if match:
        return int(match.group())

    return float("inf")


files = sorted(
    [f for f in os.listdir(INPUT_FOLDER)
     if f.lower().endswith(VALID_EXTENSIONS)],
    key=extract_depth_key
)

print(f"{len(files)} files found")

print("\nPROCESSING ORDER CHECK:")
for f in files[:25]:
    print(f, "->", extract_depth_key(f))

# =====================================================
# ROI MODE
# =====================================================
# Full-image mode:
#   The whole image is used as the processing ROI.
#   This assumes your exported images are already cropped/scaled as desired.

USE_FULL_IMAGE_AS_ROI = True

# =====================================================
# FUNCTIONS
# =====================================================

def circ_distance(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)


def preprocess_roi(roi_uint8, mode="bilateral_only"):
    """
    Bilateral-only preprocessing.

    This version intentionally removes CLAHE from the production workflow
    because your ablation showed that bilateral-only gave the most defensible
    results for the current validation stage.
    """

    if mode != "bilateral_only":
        raise ValueError(
            f"This V10 script only supports mode='bilateral_only'. Got: {mode}"
        )

    roi_proc = cv2.bilateralFilter(
        roi_uint8,
        d=BILATERAL_D,
        sigmaColor=BILATERAL_SIGMA_COLOR,
        sigmaSpace=BILATERAL_SIGMA_SPACE
    )

    roi_eq = roi_proc.copy()

    return roi_eq, roi_proc


def fit_predict_kmeans_1d(image_uint8):
    """
    Memory-safe KMeans on 1D processed acoustic amplitude.
    """
    H, W = image_uint8.shape
    pixels = image_uint8.reshape(-1, 1).astype(np.float32)

    if KMEANS_SAMPLE_PIXELS is not None and pixels.shape[0] > KMEANS_SAMPLE_PIXELS:
        rng = np.random.default_rng(KMEANS_RANDOM_STATE)
        sample_idx = rng.choice(
            pixels.shape[0],
            size=KMEANS_SAMPLE_PIXELS,
            replace=False
        )
        train_pixels = pixels[sample_idx]
    else:
        train_pixels = pixels

    kmeans = KMeans(
        n_clusters=N_CLUSTERS,
        n_init=10,
        random_state=KMEANS_RANDOM_STATE
    )
    kmeans.fit(train_pixels)

    labels_flat = kmeans.predict(pixels)
    labels = labels_flat.reshape(H, W)

    means = np.array([
        pixels[labels_flat == i].mean()
        if np.any(labels_flat == i)
        else 999
        for i in range(N_CLUSTERS)
    ])

    return labels, means


def segment_dark_pixels(image_uint8, method="kmeans"):
    """
    KMeans-only segmentation.

    Returns:
      mask_raw, labels, means
    """
    if method != "kmeans":
        raise ValueError(f"This V12 script only supports method='kmeans'. Got: {method}")

    labels, means = fit_predict_kmeans_1d(image_uint8)
    dark_cluster = int(np.argmin(means))
    mask_raw = (labels == dark_cluster).astype(np.uint8)

    return mask_raw, labels, means

def suppress_laterally_continuous_layers(mask):
    H, W = mask.shape
    row_dark_fraction = np.mean(mask, axis=1)
    layer_rows = row_dark_fraction > MAX_ROW_DARK_FRACTION
    clean_layer_rows = np.zeros_like(layer_rows, dtype=bool)

    start = None
    for yy, val in enumerate(layer_rows):
        if val and start is None:
            start = yy
        elif not val and start is not None:
            if yy - start >= MIN_LAYER_RUN:
                clean_layer_rows[start:yy] = True
            start = None

    if start is not None and H - start >= MIN_LAYER_RUN:
        clean_layer_rows[start:H] = True

    if LAYER_DILATION_Y > 1:
        kernel = np.ones(LAYER_DILATION_Y, dtype=np.uint8)
        clean_layer_rows = np.convolve(
            clean_layer_rows.astype(np.uint8),
            kernel,
            mode="same"
        ) > 0

    suppressed = mask.copy()
    suppressed[clean_layer_rows, :] = 0
    return suppressed, clean_layer_rows


def remove_vertical_line_artifacts_for_detection(mask):
    if not ENABLE_PRE_AZIMUTH_VERTICAL_LINE_SUPPRESSION:
        return mask

    H, W = mask.shape
    kernel_h = int(round(H * VERTICAL_LINE_KERNEL_HEIGHT_FRACTION))
    kernel_h = max(VERTICAL_LINE_KERNEL_MIN_HEIGHT_PX, kernel_h)
    kernel_h = min(kernel_h, H)

    if kernel_h < 3:
        return mask

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (VERTICAL_LINE_KERNEL_WIDTH_PX, kernel_h)
    )

    opened = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, kernel)
    line_mask = np.zeros_like(mask, dtype=np.uint8)

    for region in regionprops(label(opened)):
        y0, x0, y1, x1 = region.bbox
        height = y1 - y0
        width = x1 - x0

        if width > VERTICAL_LINE_MAX_BBOX_WIDTH_PX:
            continue
        if height / H < VERTICAL_LINE_MIN_HEIGHT_FRACTION:
            continue

        coords = region.coords
        line_mask[coords[:, 0], coords[:, 1]] = 1

    if VERTICAL_LINE_SUBTRACT_DILATION_X > 0 and np.sum(line_mask) > 0:
        dil_kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (2 * VERTICAL_LINE_SUBTRACT_DILATION_X + 1, 1)
        )
        line_mask = cv2.dilate(line_mask, dil_kernel, iterations=1)

    cleaned = mask.copy().astype(np.uint8)
    cleaned[line_mask > 0] = 0

    removed = int(np.sum(mask) - np.sum(cleaned))
    if removed > 0:
        print(f"PRE-AZIMUTH VERTICAL LINE SUPPRESSION | removed_pixels={removed}, kernel_h={kernel_h}")

    return cleaned


def build_verticality_weighted_mask(mask):
    H, W = mask.shape
    labels_cc = label(mask)
    weighted_mask = np.zeros_like(mask, dtype=np.float32)
    cleaned_mask = np.zeros_like(mask, dtype=np.uint8)
    edge_margin_px = int(W * EDGE_ARTIFACT_MARGIN_FRACTION)

    for region in regionprops(labels_cc):
        if region.area < 15:
            continue

        y0, x0, y1, x1 = region.bbox
        bbox_h = y1 - y0
        bbox_w = x1 - x0

        if bbox_w / W > MAX_COMPONENT_WIDTH_FRACTION:
            continue
        if bbox_w / (bbox_h + 1e-6) > MAX_HORIZONTAL_ASPECT:
            continue

        coords = region.coords
        ys = coords[:, 0]
        xs = coords[:, 1]
        height = ys.max() - ys.min() + 1
        width = xs.max() - xs.min() + 1
        fill_fraction = region.area / (bbox_h * bbox_w + 1e-6)
        near_edge = x0 <= edge_margin_px or x1 >= W - edge_margin_px

        if ENABLE_VERTICAL_ARTIFACT_REJECTION:
            very_thin = width <= MAX_THIN_COMPONENT_WIDTH_PX
            too_narrow = width < MIN_BREAKOUT_COMPONENT_WIDTH_PX
            very_tall = height / H >= MIN_ARTIFACT_HEIGHT_FRACTION
            sparse_line_like = fill_fraction <= MAX_ARTIFACT_FILL_FRACTION

            if very_tall and sparse_line_like and (very_thin or near_edge):
                continue
            if near_edge and too_narrow and very_tall:
                continue

        verticality = height / (width + 1e-6)
        verticality = np.clip(verticality / 4, 0, 1)
        orientation_deg = abs(np.rad2deg(region.orientation))
        diagonal_penalty = np.exp(-orientation_deg / 35)
        weight = 1.0 + VERTICALITY_WEIGHT * verticality * diagonal_penalty

        weighted_mask[ys, xs] = weight
        cleaned_mask[ys, xs] = 1

    return weighted_mask, cleaned_mask


def compute_azimuth_signal(mask):
    H, W = mask.shape
    persistence = np.zeros(W)
    elongation = np.zeros(W)

    for i in range(W):
        col = mask[:, i]
        if np.sum(col) == 0:
            continue

        diff = np.diff(np.concatenate(([0], col, [0])))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        n = min(len(starts), len(ends))
        if n == 0:
            continue

        runs = ends[:n] - starts[:n]
        persistence[i] = np.max(runs) / H
        elongation[i] = np.mean(col)

    persistence_s = gaussian_filter1d(persistence, sigma=4)
    elongation_s = gaussian_filter1d(elongation, sigma=4)
    signal = 0.35 * persistence_s + 0.65 * elongation_s
    signal = gaussian_filter1d(signal, sigma=7)
    return signal


def reject_artifact_like_azimuth_columns(mask, signal):
    if not ENABLE_VERTICAL_ARTIFACT_REJECTION:
        return signal

    H, W = mask.shape
    corrected = signal.copy()
    col_density = np.mean(mask, axis=0)

    for x in range(W):
        col = mask[:, x]
        if np.sum(col) == 0:
            continue

        diff = np.diff(np.concatenate(([0], col, [0])))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        if len(starts) == 0 or len(ends) == 0:
            continue

        runs = ends - starts
        longest_fraction = np.max(runs) / H
        left = max(0, x - 2)
        right = min(W, x + 3)
        local_support = np.mean(col_density[left:right])
        near_edge = (
            x < int(W * EDGE_ARTIFACT_MARGIN_FRACTION) or
            x > int(W * (1 - EDGE_ARTIFACT_MARGIN_FRACTION))
        )

        if longest_fraction > MIN_ARTIFACT_HEIGHT_FRACTION and local_support < 0.08:
            corrected[x] *= 0.15
        if near_edge and longest_fraction > 0.10:
            corrected[x] *= 0.10

    return corrected


def find_active_regions(signal, W):
    threshold = np.percentile(signal, HOTSPOT_PERCENTILE)
    active = signal > threshold
    regions = []
    start = None

    for i in range(W):
        if active[i] and start is None:
            start = i
        elif not active[i] and start is not None:
            regions.append((start, i))
            start = None

    if start is not None:
        regions.append((start, W - 1))

    return regions, threshold


def build_candidates(regions, signal, weighted_mask, W):
    candidates = []

    for L, R in regions:
        width_px = R - L
        width_deg = 360 * width_px / W

        if width_deg < MIN_REGION_WIDTH_DEG:
            continue

        phi = (L + R) / 2
        phi_deg = 360 * phi / W
        peak_strength = np.mean(signal[L:R])
        verticality_score = np.mean(weighted_mask[:, L:R])
        final_strength = peak_strength * verticality_score

        candidates.append({
            "L": L,
            "R": R,
            "phi": phi_deg,
            "Wdeg": width_deg,
            "strength": final_strength
        })

    return candidates





def search_best_single_breakout(candidates, signal):
    """
    Single-breakout fallback in azimuth space.

    This is intentionally strict:
      - only near 180 degrees
      - only if the single lobe is strong
      - only if it dominates the rest of the azimuth signal

    Returns:
      best_single, best_score, metrics
    """
    if not ENABLE_SINGLE_BREAKOUT_MODE:
        return None, -1, {}

    if candidates is None or len(candidates) == 0:
        return None, -1, {}

    total_energy = float(np.sum(signal)) + 1e-6
    background = float(np.percentile(signal, 25)) + 1e-6

    best_single = None
    best_score = -1
    best_metrics = {}

    for c in candidates:
        phi = float(c["phi"])
        width_deg = float(c["Wdeg"])
        strength = float(c["strength"])

        azimuth_error = circ_distance(phi, SINGLE_BREAKOUT_TARGET_AZIMUTH_DEG)

        if azimuth_error > SINGLE_BREAKOUT_AZIMUTH_TOLERANCE_DEG:
            continue

        if width_deg > MAX_SINGLE_BREAKOUT_WIDTH_DEG:
            continue

        if strength < MIN_SINGLE_BREAKOUT_STRENGTH:
            continue

        L = int(c["L"])
        R = int(c["R"])
        if R <= L:
            continue

        local_signal = candidate_signal_slice(signal, c)
        if local_signal.size == 0:
            continue

        effective_peak = (
            EFFECTIVE_PEAK_P90_WEIGHT * float(np.percentile(local_signal, 90)) +
            EFFECTIVE_PEAK_MEAN_WEIGHT * float(np.mean(local_signal))
        )

        dominance = float(np.sum(local_signal)) / total_energy
        peak_bg_ratio = effective_peak / background

        other = signal.copy().astype(float)
        other = _zero_signal_interval(other, L, R)
        strongest_other = float(np.max(other)) + 1e-6
        isolation = effective_peak / strongest_other

        print(
            f"SINGLE QC | phi={phi:.1f}, width={width_deg:.1f}, "
            f"az_error={azimuth_error:.1f}, strength={strength:.5f}, "
            f"dominance={dominance:.2f}, peak_bg={peak_bg_ratio:.2f}, "
            f"isolation={isolation:.2f}"
        )

        if dominance < MIN_SINGLE_BREAKOUT_DOMINANCE:
            continue

        if peak_bg_ratio < MIN_SINGLE_BREAKOUT_PEAK_BG_RATIO:
            continue

        if isolation < MIN_SINGLE_BREAKOUT_ISOLATION:
            continue

        azimuth_score = np.exp(-(azimuth_error ** 2) / (2 * 25 ** 2))
        score = strength * (0.60 + 0.40 * azimuth_score) * (0.70 + 0.30 * dominance) * min(isolation, 3.0)

        if score > best_score:
            best_score = score
            best_single = c.copy()
            best_metrics = {
                "single_azimuth_error_deg": float(azimuth_error),
                "single_dominance": float(dominance),
                "single_peak_bg_ratio": float(peak_bg_ratio),
                "single_isolation": float(isolation),
                "single_effective_peak": float(effective_peak),
                "single_score": float(score)
            }

    return best_single, best_score, best_metrics



def search_continuity_single_rescue(candidates, signal, confirmed_direction, previous_image_positive):
    """
    Continuity fallback using only confirmed_direction as memory.

    If the immediately previous image was positive and the current image has
    a candidate lobe close to the last accepted breakout azimuth, accept it as
    a continuation even when the classic pair/single gates fail.

    This is intentionally softer than search_best_single_breakout() because it
    is not a first-pass detector; it is a depth-continuity rescue.
    """
    if not ENABLE_CONTINUITY_SINGLE_RESCUE:
        return None, -1, {}

    if not previous_image_positive:
        return None, -1, {}

    if confirmed_direction is None:
        return None, -1, {}

    if candidates is None or len(candidates) == 0:
        return None, -1, {}

    background = float(np.percentile(signal, 25)) + 1e-6
    best = None
    best_score = -1
    best_metrics = {}

    for c in candidates:
        phi = float(c["phi"])
        width_deg = float(c["Wdeg"])
        strength = float(c["strength"])
        drift = circ_distance(phi, confirmed_direction)

        if drift > CONTINUITY_SINGLE_RESCUE_TOL_DEG:
            continue

        if width_deg > MAX_CONTINUITY_SINGLE_WIDTH_DEG:
            continue

        if strength < MIN_CONTINUITY_SINGLE_RESCUE_STRENGTH:
            continue

        L = int(c["L"])
        R = int(c["R"])
        if R <= L:
            continue

        local_signal = candidate_signal_slice(signal, c)
        if local_signal.size == 0:
            continue

        effective_peak = (
            EFFECTIVE_PEAK_P90_WEIGHT * float(np.percentile(local_signal, 90)) +
            EFFECTIVE_PEAK_MEAN_WEIGHT * float(np.mean(local_signal))
        )

        peak_bg_ratio = effective_peak / background

        if peak_bg_ratio < MIN_CONTINUITY_SINGLE_PEAK_BG_RATIO:
            continue

        drift_score = np.exp(-(drift ** 2) / (2 * 35 ** 2))
        score = strength * (0.50 + 0.50 * drift_score) * min(peak_bg_ratio, 3.0)

        print(
            f"CONTINUITY SINGLE QC | phi={phi:.1f}, "
            f"prior={confirmed_direction:.1f}, drift={drift:.1f}, "
            f"width={width_deg:.1f}, strength={strength:.5f}, "
            f"peak_bg={peak_bg_ratio:.2f}, score={score:.5f}"
        )

        if score > best_score:
            best_score = score
            best = c.copy()
            best_metrics = {
                "continuity_rescue_used": True,
                "continuity_prior_direction_deg": float(confirmed_direction),
                "continuity_drift_deg": float(drift),
                "continuity_peak_bg_ratio": float(peak_bg_ratio),
                "continuity_score": float(score),
            }

    return best, best_score, best_metrics

def _zero_signal_interval(arr, L, R):
    """
    Zeroes an interval in a 1D circular signal helper copy.
    Handles both normal and wrapped 0/360 intervals.
    """
    W = len(arr)

    L = int(max(0, min(L, W)))
    R = int(max(0, min(R, W)))

    if R > L:
        arr[L:R] = 0
    elif R < L:
        arr[L:W] = 0
        arr[0:R] = 0

    return arr


def pair_valley_quality(signal, c1, c2):
    """
    Clean valley QC using effective lobe strength:
        weak_peak = min(0.6*P90 + 0.4*mean for each candidate window)

    Removes the over-strict V16 valley-width / plateau-mean logic.
    """

    W = len(signal)

    p1 = int(round((c1["L"] + c1["R"]) / 2))
    p2 = int(round((c2["L"] + c2["R"]) / 2))

    p1 = int(np.clip(p1, 0, W - 1))
    p2 = int(np.clip(p2, 0, W - 1))

    s1 = candidate_signal_slice(signal, c1)
    s2 = candidate_signal_slice(signal, c2)

    empty_metrics = {
        "valley_depth": np.nan,
        "valley_noise": np.nan,
        "peak_bg_ratio": np.nan,
        "pair_isolation": np.nan,
        "valley_min": np.nan,
        "weak_effective_peak": np.nan,
        "strongest_other": np.nan
    }

    if s1.size == 0 or s2.size == 0:
        return False, empty_metrics

    effective1 = (
        EFFECTIVE_PEAK_P90_WEIGHT * float(np.percentile(s1, 90)) +
        EFFECTIVE_PEAK_MEAN_WEIGHT * float(np.mean(s1))
    )

    effective2 = (
        EFFECTIVE_PEAK_P90_WEIGHT * float(np.percentile(s2, 90)) +
        EFFECTIVE_PEAK_MEAN_WEIGHT * float(np.mean(s2))
    )

    weak_peak = min(effective1, effective2)

    if weak_peak <= 0:
        return False, empty_metrics

    a, b = sorted([p1, p2])
    direct = signal[a:b + 1]
    wrap = np.concatenate([signal[b:], signal[:a + 1]])
    valley_path = direct if len(direct) <= len(wrap) else wrap

    if valley_path.size < 5:
        return False, empty_metrics

    valley_min = float(np.min(valley_path))
    valley_std = float(np.std(valley_path))
    background = float(np.percentile(signal, 25))

    valley_depth = (weak_peak - valley_min) / (weak_peak + 1e-6)
    valley_noise = valley_std / (weak_peak + 1e-6)
    peak_bg_ratio = weak_peak / (background + 1e-6)

    other = signal.copy().astype(float)
    other = _zero_signal_interval(other, c1["L"], c1["R"])
    other = _zero_signal_interval(other, c2["L"], c2["R"])

    strongest_other = float(np.max(other)) if other.size > 0 else 0.0
    pair_isolation = weak_peak / (strongest_other + 1e-6)

    valley_ok = (
        valley_depth >= MIN_INTERPEAK_VALLEY_DEPTH
        and valley_noise <= MAX_INTERPEAK_VALLEY_NOISE
        and peak_bg_ratio >= MIN_PEAK_TO_BACKGROUND_RATIO
    )

    if ENABLE_PAIR_ISOLATION_GATE:
        valley_ok = valley_ok and pair_isolation >= MIN_PAIR_ISOLATION

    metrics = {
        "valley_depth": float(valley_depth),
        "valley_noise": float(valley_noise),
        "peak_bg_ratio": float(peak_bg_ratio),
        "pair_isolation": float(pair_isolation),
        "valley_min": float(valley_min),
        "weak_effective_peak": float(weak_peak),
        "strongest_other": float(strongest_other)
    }

    return bool(valley_ok), metrics



def circular_interval_mask(W, L, R):
    """
    Boolean mask for a candidate interval in azimuth-signal index space.
    Handles both normal and wrapped 0/360 intervals.
    """
    m = np.zeros(W, dtype=bool)

    L = int(max(0, min(L, W)))
    R = int(max(0, min(R, W)))

    if R > L:
        m[L:R] = True
    elif R < L:
        m[L:W] = True
        m[0:R] = True

    return m


def candidate_signal_slice(signal, c):
    L = int(c["L"])
    R = int(c["R"])
    W = len(signal)

    if R > L:
        return signal[L:R]

    # wrapped interval across 360/0
    return np.concatenate([
        signal[L:W],
        signal[0:R]
    ])

def pair_signal_balance_metrics(c1, c2, signal):
    """
    Pair-level 1D signal metrics used to reject false pairs.

    Handles wrapped candidates across 0/360 correctly.
    """
    W = len(signal)

    L1, R1 = int(c1["L"]), int(c1["R"])
    L2, R2 = int(c2["L"]), int(c2["R"])

    s1 = candidate_signal_slice(signal, c1)
    s2 = candidate_signal_slice(signal, c2)

    energy1 = float(np.sum(s1))
    energy2 = float(np.sum(s2))

    strength1 = float(c1["strength"])
    strength2 = float(c2["strength"])

    width1 = float(c1["Wdeg"])
    width2 = float(c2["Wdeg"])

    effective1 = (
        EFFECTIVE_PEAK_P90_WEIGHT * float(np.percentile(s1, 90)) +
        EFFECTIVE_PEAK_MEAN_WEIGHT * float(np.mean(s1))
    ) if s1.size > 0 else 0.0

    effective2 = (
        EFFECTIVE_PEAK_P90_WEIGHT * float(np.percentile(s2, 90)) +
        EFFECTIVE_PEAK_MEAN_WEIGHT * float(np.mean(s2))
    ) if s2.size > 0 else 0.0

    weak_effective_peak = min(effective1, effective2)
    weak_energy = min(energy1, energy2)

    used = (
        circular_interval_mask(W, L1, R1)
        | circular_interval_mask(W, L2, R2)
    )

    other_signal = np.asarray(signal, dtype=float).copy()
    other_signal[used] = 0.0
    strongest_other = float(np.max(other_signal)) if other_signal.size > 0 else 0.0

    other_candidate_energies = []

    for c in (globals().get("_CURRENT_PAIR_CANDIDATES_FOR_QC") or []):

        # Skip the accepted pair objects themselves when object identity is preserved.
        if c is c1 or c is c2:
            continue

        L, R = int(c["L"]), int(c["R"])
        cm = circular_interval_mask(W, L, R)

        # Skip any candidate window overlapping the accepted pair windows.
        if np.any(cm & used):
            continue

        other_candidate_energies.append(
            float(np.sum(candidate_signal_slice(signal, c)))
        )

    strongest_other_energy = (
        max(other_candidate_energies)
        if len(other_candidate_energies) > 0
        else 0.0
    )

    return {
        "strength_balance": min(strength1, strength2) / (max(strength1, strength2) + 1e-6),
        "energy_balance": min(energy1, energy2) / (max(energy1, energy2) + 1e-6),
        "width_balance": min(width1, width2) / (max(width1, width2) + 1e-6),
        "energy1": energy1,
        "energy2": energy2,
        "weak_energy": weak_energy,
        "strongest_other": strongest_other,
        "strongest_other_energy": strongest_other_energy,
        "weak_effective_peak": weak_effective_peak,
        "competing_peak_to_weak": strongest_other / (weak_effective_peak + 1e-6),
        "competing_energy_to_weak": strongest_other_energy / (weak_energy + 1e-6),
    }


def candidate_is_edge_spike(c):
    """
    True when a candidate is a narrow spike at the 0/360 image boundary.
    """
    phi = float(c["phi"])
    width = float(c["Wdeg"])
    near_edge = (
        phi <= EDGE_SPIKE_AZIMUTH_MARGIN_DEG
        or phi >= 360.0 - EDGE_SPIKE_AZIMUTH_MARGIN_DEG
    )
    return near_edge and width <= MAX_EDGE_SPIKE_WIDTH_DEG
def merge_wraparound_candidates(candidates, W, edge_merge_deg=35):
    """
    Merge candidates that are split across the 0/360 boundary.

    Example:
      candidate A near 350°
      candidate B near 10°

    becomes one wrapped candidate centered near 0°.
    """

    if candidates is None or len(candidates) < 2:
        return candidates

    left = None
    right = None

    for c in candidates:
        if c["phi"] <= edge_merge_deg:
            left = c
        elif c["phi"] >= 360 - edge_merge_deg:
            right = c

    if left is None or right is None:
        return candidates

    merged = {}

    merged["L"] = int(right["L"])
    merged["R"] = int(left["R"])
    merged["phi"] = 0.0

    merged["Wdeg"] = float(left["Wdeg"] + right["Wdeg"])
    merged["strength"] = max(float(left["strength"]), float(right["strength"]))

    new_candidates = []

    for c in candidates:
        if c is left or c is right:
            continue
        new_candidates.append(c)

    new_candidates.append(merged)

    print(
        f"MERGED WRAPAROUND CANDIDATES | "
        f"phi_left={left['phi']:.1f}, phi_right={right['phi']:.1f}, "
        f"merged_width={merged['Wdeg']:.1f}"
    )

    return new_candidates


def search_best_pair(candidates, signal, confirmed_direction):
    best_pair = None
    best_score = -1
    total_energy = np.sum(signal)

    # Used by pair_signal_balance_metrics() to compare the accepted pair
    # against other candidate windows.
    global _CURRENT_PAIR_CANDIDATES_FOR_QC
    _CURRENT_PAIR_CANDIDATES_FOR_QC = candidates

    # Counts pairs that survive all pair-level gates and reach scoring.
    # In cave-rich mode, too many plausible pairs usually indicates
    # multimodal cave/megapore clutter rather than one coherent breakout pair.
    valid_pair_count = 0

    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):

            c1 = candidates[i]
            c2 = candidates[j]

            separation = circ_distance(c1["phi"], c2["phi"])

            if not (MIN_AZIMUTH_SEPARATION <= separation <= MAX_AZIMUTH_SEPARATION):
                continue

            width_ratio = c1["Wdeg"] / (c2["Wdeg"] + 1e-6)

            if not (MIN_WIDTH_RATIO <= width_ratio <= MAX_WIDTH_RATIO):
                continue

            weak = min(c1["strength"], c2["strength"])
            strong = max(c1["strength"], c2["strength"])
            peak_ratio = weak / (strong + 1e-6)

            if weak < MIN_WEAK_LOBE_STRENGTH:
                continue

            pair_energy = (
                np.sum(candidate_signal_slice(signal, c1)) +
                np.sum(candidate_signal_slice(signal, c2))
            )
            dominance = pair_energy / (total_energy + 1e-6)

            if dominance < MIN_PAIR_DOMINANCE:
                continue

            balance = pair_signal_balance_metrics(c1, c2, signal)

            if ENABLE_PAIR_BALANCE_GATES:
                print(
                    f"PAIR BALANCE QC | "
                    f"strength_balance={balance['strength_balance']:.2f}, "
                    f"energy_balance={balance['energy_balance']:.2f}, "
                    f"width_balance={balance['width_balance']:.2f}, "
                    f"comp_peak/weak={balance['competing_peak_to_weak']:.2f}, "
                    f"comp_energy/weak={balance['competing_energy_to_weak']:.2f}"
                )

                if balance["strength_balance"] < MIN_PAIR_STRENGTH_BALANCE:
                    print(
                        f"REJECTED: weak paired lobe | "
                        f"strength_balance={balance['strength_balance']:.2f}, "
                        f"min_required={MIN_PAIR_STRENGTH_BALANCE:.2f}"
                    )
                    continue

                asymmetric_pair_rescue = (
                    IMAGE_REGIME == "cave_rich"
                    and separation >= 155
                    and balance["strength_balance"] >= 0.45
                    and balance["energy_balance"] >= 0.15
                    and balance["width_balance"] >= 0.20
                    and balance["competing_peak_to_weak"] <= 1.05
                )

                if (
                    balance["energy_balance"] < MIN_PAIR_ENERGY_BALANCE
                    and not asymmetric_pair_rescue
                ):
                    print(
                        f"REJECTED: poor pair energy balance | "
                        f"energy_balance={balance['energy_balance']:.2f}, "
                        f"min_required={MIN_PAIR_ENERGY_BALANCE:.2f}"
                    )
                    continue

                if asymmetric_pair_rescue:
                    print(
                        f"ASYMMETRIC PAIR RESCUE | "
                        f"sep={separation:.1f}, "
                        f"strength_balance={balance['strength_balance']:.2f}, "
                        f"energy_balance={balance['energy_balance']:.2f}, "
                        f"width_balance={balance['width_balance']:.2f}"
                    )

                if balance["width_balance"] < MIN_PAIR_WIDTH_BALANCE:
                    print(
                        f"REJECTED: poor pair width balance | "
                        f"width_balance={balance['width_balance']:.2f}, "
                        f"min_required={MIN_PAIR_WIDTH_BALANCE:.2f}"
                    )
                    continue

            if ENABLE_EDGE_SPIKE_PAIR_VETO:
                edge_spike_pair = (
                    candidate_is_edge_spike(c1)
                    or candidate_is_edge_spike(c2)
                )

                if edge_spike_pair:
                    print(
                        f"REJECTED: edge spike paired as breakout | "
                        f"phi1={c1['phi']:.1f}, width1={c1['Wdeg']:.1f}, "
                        f"phi2={c2['phi']:.1f}, width2={c2['Wdeg']:.1f}"
                    )
                    continue

            if ENABLE_COMPETING_PEAK_VETO:
                if balance["competing_peak_to_weak"] > MAX_COMPETING_PEAK_TO_WEAK_PEAK_RATIO:
                    print(
                        f"REJECTED: competing third peak | "
                        f"competing_peak_to_weak={balance['competing_peak_to_weak']:.2f}, "
                        f"max_allowed={MAX_COMPETING_PEAK_TO_WEAK_PEAK_RATIO:.2f}"
                    )
                    continue

                if balance["competing_energy_to_weak"] > MAX_COMPETING_ENERGY_TO_WEAK_ENERGY_RATIO:
                    print(
                        f"REJECTED: competing third-lobe energy | "
                        f"competing_energy_to_weak={balance['competing_energy_to_weak']:.2f}, "
                        f"max_allowed={MAX_COMPETING_ENERGY_TO_WEAK_ENERGY_RATIO:.2f}"
                    )
                    continue

            valley_metrics = {}

            if ENABLE_VALLEY_GATE:

                valley_ok, valley_metrics = pair_valley_quality(signal, c1, c2)

                print(
                    f"VALLEY QC | "
                    f"depth={valley_metrics.get('valley_depth', np.nan):.2f}, "
                    f"noise={valley_metrics.get('valley_noise', np.nan):.2f}, "
                    f"peak_bg={valley_metrics.get('peak_bg_ratio', np.nan):.2f}, "
                    f"isolation={valley_metrics.get('pair_isolation', np.nan):.2f}"
                )

                pair_has_wrapped_candidate = (
                    int(c1["R"]) < int(c1["L"])
                    or int(c2["R"]) < int(c2["L"])
                )

                if (
                    pair_has_wrapped_candidate
                    and balance["energy_balance"] >= 0.60
                ):
                    print(
                        "WRAPAROUND RESCUE | "
                        f"energy_balance={balance['energy_balance']:.2f}"
                    )
                    valley_ok = True

                if not valley_ok:
                    print("REJECTED: weak valley or multimodal signal")
                    continue

                # Cave-rich weak-evidence veto:
                # At this point the pair passed valley QC, but in cave-rich
                # intervals low dominance + low peak/background + weak isolation
                # is a common false-positive signature.
                if IMAGE_REGIME == "cave_rich" and ENABLE_CAVE_RICH_WEAK_EVIDENCE_VETO:

                    weak_cave_pair = (
                        dominance < CAVE_RICH_MIN_PAIR_DOMINANCE
                        and valley_metrics.get("peak_bg_ratio", 0) < CAVE_RICH_MIN_PEAK_BG_RATIO
                        and valley_metrics.get("pair_isolation", 0) < CAVE_RICH_MIN_PAIR_ISOLATION
                    )

                    if weak_cave_pair:
                        print(
                            f"REJECTED: cave-rich weak evidence pair | "
                            f"dominance={dominance:.2f}, "
                            f"peak_bg={valley_metrics.get('peak_bg_ratio', 0):.2f}, "
                            f"isolation={valley_metrics.get('pair_isolation', 0):.2f}"
                        )
                        continue

                if ENABLE_BROAD_PLATEAU_VETO:
                    broad_plateau_pair = (
                        separation < BROAD_PLATEAU_MAX_SEPARATION
                        and max(c1["Wdeg"], c2["Wdeg"]) > BROAD_PLATEAU_MIN_WIDTH_DEG
                    )

                    if broad_plateau_pair:
                        print(
                            f"REJECTED: broad plateau | "
                            f"sep={separation:.1f}, "
                            f"max_width={max(c1['Wdeg'], c2['Wdeg']):.1f}, "
                            f"sep_limit={BROAD_PLATEAU_MAX_SEPARATION}, "
                            f"width_limit={BROAD_PLATEAU_MIN_WIDTH_DEG}"
                        )
                        continue

                # Cave-rich edge veto:
                # rejects 0/360-boundary spikes paired with a weak interior lobe,
                # but allows near-opposite edge-wrapped true pairs.
                if IMAGE_REGIME == "cave_rich":

                    edge_margin = 25

                    edge_dominated = (
                        c1["phi"] < edge_margin
                        or c1["phi"] > (360 - edge_margin)
                        or c2["phi"] < edge_margin
                        or c2["phi"] > (360 - edge_margin)
                    )

                    weak_partner = (
                        min(c1["strength"], c2["strength"])
                        <
                        0.60 * max(c1["strength"], c2["strength"])
                    )

                    allow_edge_pair = (
                        separation >= 155
                        and balance["energy_balance"] >= 0.25
                        and valley_metrics.get("valley_depth", 0) >= 0.45
                    )

                    if edge_dominated and weak_partner and not allow_edge_pair:
                        print("REJECTED: cave-rich edge-dominated asymmetric pair")
                        continue

                c1["valley_depth"] = valley_metrics.get("valley_depth", np.nan)
                c2["valley_depth"] = valley_metrics.get("valley_depth", np.nan)

                c1["valley_noise"] = valley_metrics.get("valley_noise", np.nan)
                c2["valley_noise"] = valley_metrics.get("valley_noise", np.nan)

                c1["peak_bg_ratio"] = valley_metrics.get("peak_bg_ratio", np.nan)
                c2["peak_bg_ratio"] = valley_metrics.get("peak_bg_ratio", np.nan)

                c1["pair_isolation"] = valley_metrics.get("pair_isolation", np.nan)
                c2["pair_isolation"] = valley_metrics.get("pair_isolation", np.nan)

                c1["weak_effective_peak"] = valley_metrics.get("weak_effective_peak", np.nan)
                c2["weak_effective_peak"] = valley_metrics.get("weak_effective_peak", np.nan)

            # Keep QC metrics in the winning candidate dictionaries.
            c1["pair_strength_balance"] = balance["strength_balance"]
            c2["pair_strength_balance"] = balance["strength_balance"]
            c1["pair_energy_balance"] = balance["energy_balance"]
            c2["pair_energy_balance"] = balance["energy_balance"]
            c1["pair_width_balance"] = balance["width_balance"]
            c2["pair_width_balance"] = balance["width_balance"]
            c1["competing_peak_to_weak"] = balance["competing_peak_to_weak"]
            c2["competing_peak_to_weak"] = balance["competing_peak_to_weak"]
            c1["competing_energy_to_weak"] = balance["competing_energy_to_weak"]
            c2["competing_energy_to_weak"] = balance["competing_energy_to_weak"]

            pair_direction = (c1["phi"] + c2["phi"]) / 2

            if confirmed_direction is not None:
                drift = circ_distance(pair_direction, confirmed_direction)

                if drift > MAX_DIRECTION_DRIFT:
                    print(
                        f"REJECTED: direction drift | "
                        f"pair_direction={pair_direction:.1f}, "
                        f"confirmed={confirmed_direction:.1f}, "
                        f"drift={drift:.1f}, "
                        f"max={MAX_DIRECTION_DRIFT}"
                    )
                    continue

            symmetry_score = np.exp(-((separation - 180) ** 2) / (2 * 65 ** 2))
            width_score = min(width_ratio, 1 / (width_ratio + 1e-6))
            strength_score = c1["strength"] + c2["strength"]

            pair_score = (
                strength_score *
                (0.45 + 0.55 * symmetry_score) *
                (0.85 + 0.15 * min(peak_ratio, 1.0)) *
                (0.70 + 0.30 * width_score) *
                (0.70 + 0.30 * dominance)
            )

            valid_pair_count += 1

            print(
                f"PAIR CANDIDATE | "
                f"phi1={c1['phi']:.1f}, "
                f"phi2={c2['phi']:.1f}, "
                f"sep={separation:.1f}, "
                f"peak_ratio={peak_ratio:.2f}, "
                f"dominance={dominance:.2f}, "
                f"score={pair_score:.5f}"
            )

            if pair_score > best_score:
                best_score = pair_score
                best_pair = (c1.copy(), c2.copy())

    if (
        IMAGE_REGIME == "cave_rich"
        and ENABLE_CAVE_RICH_MULTIPAIR_VETO
        and valid_pair_count > MAX_CAVE_RICH_VALID_PAIRS
    ):

        print(
            f"REJECTED: cave-rich multimodal signal with "
            f"{valid_pair_count} valid pairs"
        )

        return None, best_score

    return best_pair, best_score


def get_manual_windows(H):
    window_height = min(WINDOW_HEIGHT, H)
    window_step = min(WINDOW_STEP, window_height)
    print(f"ROI height={H} px | window height={window_height} px | step={window_step} px")
    return window_height, window_step


def local_window_is_valid(
    local,
    density_threshold,
    min_component_area,
    min_verticality
):
    """
    Fast local validation.

    Avoids expensive full regionprops unless the window passes cheap tests.
    """

    if local.size == 0:
        return False

    pixel_count = int(np.sum(local > 0))

    if pixel_count == 0:
        return False

    density = pixel_count / local.size

    if density < density_threshold:
        return False

    # Fast column/row support check before connected components
    rows_active = np.where(np.any(local > 0, axis=1))[0]
    cols_active = np.where(np.any(local > 0, axis=0))[0]

    if rows_active.size == 0 or cols_active.size == 0:
        return False

    height = rows_active.max() - rows_active.min() + 1
    width = cols_active.max() - cols_active.min() + 1

    if pixel_count < min_component_area:
        return False

    approx_verticality = height / (width + 1e-6)

    # If very permissive verticality is being used, this is enough
    if approx_verticality >= min_verticality:
        return True

    return False


def validate_vertical_windows(mask, accepted_windows, window_height, window_step):
    H, W = mask.shape
    final_mask = np.zeros_like(mask, dtype=np.uint8)
    hotspot_boxes = []

    if H <= window_height:
        y_starts = [0]
    else:
        y_starts = list(range(0, H - window_height + 1, window_step))
        if y_starts[-1] != H - window_height:
            y_starts.append(H - window_height)

    strict_valid = {}
    weak_valid = {}

    for win_idx, (L, R) in enumerate(accepted_windows):
        for y0 in y_starts:
            y1 = min(y0 + window_height, H)
            local = mask[y0:y1, L:R]

            strict_ok = local_window_is_valid(
                local,
                LOCAL_BREAKOUT_THRESHOLD,
                MIN_LOCAL_COMPONENT_AREA,
                MIN_LOCAL_VERTICALITY
            )
            weak_ok = local_window_is_valid(
                local,
                WEAK_LOCAL_BREAKOUT_THRESHOLD,
                WEAK_MIN_LOCAL_COMPONENT_AREA,
                WEAK_MIN_LOCAL_VERTICALITY
            )

            strict_valid[(win_idx, y0)] = strict_ok
            weak_valid[(win_idx, y0)] = weak_ok

            if strict_ok:
                final_mask[y0:y1, L:R] = local
                hotspot_boxes.append((L, y0, R, y1))

    if ENABLE_MIRROR_ASSISTED_VALIDATION and len(accepted_windows) == 2:
        for y0 in y_starts:
            y1 = min(y0 + window_height, H)
            for source_idx, target_idx in [(0, 1), (1, 0)]:
                source_strict = strict_valid.get((source_idx, y0), False)
                target_strict = strict_valid.get((target_idx, y0), False)
                target_weak = weak_valid.get((target_idx, y0), False)

                if source_strict and not target_strict and target_weak:
                    L, R = accepted_windows[target_idx]
                    final_mask[y0:y1, L:R] = mask[y0:y1, L:R]
                    hotspot_boxes.append((L, y0, R, y1))

    if ENABLE_ACCEPTED_PAIR_RESCUE and len(accepted_windows) == 2:
        for L, R in accepted_windows:
            already = np.sum(final_mask[:, L:R])
            available = mask[:, L:R]

            if np.mean(available) < PAIR_RESCUE_MIN_COLUMN_DENSITY:
                continue

            if already < PAIR_RESCUE_MIN_COMPONENT_AREA:
                rescue = np.zeros_like(available, dtype=np.uint8)
                for region in regionprops(label(available)):
                    if region.area < PAIR_RESCUE_MIN_COMPONENT_AREA:
                        continue
                    coords = region.coords
                    rescue[coords[:, 0], coords[:, 1]] = 1
                final_mask[:, L:R] = np.maximum(final_mask[:, L:R], rescue)

    return final_mask, hotspot_boxes


def expand_breakout_width(mask_local, img_local):
    if not ENABLE_WIDTH_EXPANSION:
        return mask_local

    expanded = mask_local.copy()
    H, W = mask_local.shape

    for y in range(H):
        xs = np.where(mask_local[y] > 0)[0]
        if xs.size == 0:
            continue

        xL = xs.min()
        xR = xs.max()
        core_values = img_local[y, xs]
        threshold = np.mean(core_values) + WIDTH_EXPANSION_INTENSITY_RELAX

        steps = 0
        x = xL - 1
        while x >= 0 and steps < WIDTH_EXPANSION_PX:
            if img_local[y, x] > threshold:
                break
            expanded[y, x] = 1
            x -= 1
            steps += 1

        steps = 0
        x = xR + 1
        while x < W and steps < WIDTH_EXPANSION_PX:
            if img_local[y, x] > threshold:
                break
            expanded[y, x] = 1
            x += 1
            steps += 1

    return expanded


def adaptive_row_expansion_from_seed(mask_local, img_local):
    if not ENABLE_ADAPTIVE_ROW_EXPANSION:
        return mask_local

    expanded = mask_local.copy()
    H, W = mask_local.shape

    for y in range(H):
        xs = np.where(mask_local[y] > 0)[0]
        if xs.size < ADAPTIVE_EXPANSION_MIN_SEED_PIXELS_PER_ROW:
            continue

        xL = int(xs.min())
        xR = int(xs.max())
        seed_values = img_local[y, xs]
        row_threshold = float(np.median(seed_values) + ADAPTIVE_EXPANSION_RELAX)

        steps = 0
        x = xL - 1
        while x >= 0 and steps < ADAPTIVE_EXPANSION_MAX_PX:
            if img_local[y, x] > row_threshold:
                break
            expanded[y, x] = 1
            x -= 1
            steps += 1

        steps = 0
        x = xR + 1
        while x < W and steps < ADAPTIVE_EXPANSION_MAX_PX:
            if img_local[y, x] > row_threshold:
                break
            expanded[y, x] = 1
            x += 1
            steps += 1

    return expanded


def expand_dark_fill_in_accepted_windows(final_mask, roi_image, accepted_windows, layer_rows):
    if not ENABLE_ACCEPTED_WINDOW_DARK_FILL:
        return final_mask

    H, W = final_mask.shape
    expanded_final = final_mask.copy().astype(np.uint8)

    for L, R in accepted_windows:
        Lg = max(0, L - LABEL_GROW_AZIMUTH_MARGIN_PX)
        Rg = min(W, R + LABEL_GROW_AZIMUTH_MARGIN_PX)

        seed_global = final_mask[:, L:R]
        if np.sum(seed_global) == 0:
            continue

        if LABEL_GROW_USE_FULL_HEIGHT:
            y0, y1 = 0, H
        else:
            ys = np.where(seed_global > 0)[0]
            if ys.size == 0:
                continue
            y0 = max(0, int(ys.min()) - LABEL_GROW_VERTICAL_MARGIN_PX)
            y1 = min(H, int(ys.max()) + LABEL_GROW_VERTICAL_MARGIN_PX + 1)

        img_local = roi_image[y0:y1, Lg:Rg]
        seed_local = np.zeros_like(img_local, dtype=np.uint8)
        seed_local[:, (L - Lg):(R - Lg)] = final_mask[y0:y1, L:R]

        dark_threshold = np.percentile(img_local, DARK_FILL_PERCENTILE)
        dark = (img_local <= dark_threshold).astype(np.uint8)

        vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (DARK_FILL_HORIZONTAL_KERNEL, DARK_FILL_VERTICAL_KERNEL))
        dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, vert_kernel)
        dark = remove_small_objects(dark.astype(bool), min_size=DARK_FILL_MIN_OBJECT_SIZE).astype(np.uint8)

        if DARK_FILL_REQUIRE_SEED_CONNECTION:
            seed_kernel = cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (DARK_FILL_SEED_DILATION_X, DARK_FILL_SEED_DILATION_Y)
            )
            seed_dil = cv2.dilate(seed_local.astype(np.uint8), seed_kernel, iterations=1)
            connected = np.zeros_like(dark, dtype=np.uint8)

            for region in regionprops(label(dark)):
                coords = region.coords
                touch_count = int(np.sum(seed_dil[coords[:, 0], coords[:, 1]] > 0))
                if touch_count >= DARK_FILL_MIN_SEED_PIXELS_PER_COMPONENT:
                    connected[coords[:, 0], coords[:, 1]] = 1
            dark = connected

        grown = np.maximum(seed_local, dark).astype(np.uint8)
        grown = expand_breakout_width(grown, img_local)
        grown = adaptive_row_expansion_from_seed(grown, img_local)

        expanded_final[y0:y1, Lg:Rg] = np.maximum(expanded_final[y0:y1, Lg:Rg], grown)

    return expanded_final.astype(np.uint8)


def apply_final_darkest_pixel_gate(final_mask, roi_image, accepted_windows):
    if not ENABLE_FINAL_DARKEST_PIXEL_GATE:
        return final_mask

    H, W = final_mask.shape
    gated = np.zeros_like(final_mask, dtype=np.uint8)

    for L, R in accepted_windows:
        Lg = max(0, L - FINAL_DARK_GATE_AZIMUTH_MARGIN_PX)
        Rg = min(W, R + FINAL_DARK_GATE_AZIMUTH_MARGIN_PX)

        candidate = final_mask[:, Lg:Rg]
        if np.sum(candidate) == 0:
            continue

        values = roi_image[:, Lg:Rg][candidate > 0]
        if values.size == 0:
            continue

        threshold = np.percentile(values, FINAL_DARK_PIXEL_PERCENTILE)
        dark_gate = (roi_image[:, Lg:Rg] <= threshold).astype(np.uint8)
        seed = candidate.astype(np.uint8)

        seed_kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (FINAL_GATE_SEED_KERNEL_X, FINAL_GATE_SEED_KERNEL_Y)
        )
        seed_dil = cv2.dilate(seed, seed_kernel, iterations=1)

        local_keep = np.zeros_like(candidate, dtype=np.uint8)
        for region in regionprops(label(dark_gate)):
            coords = region.coords
            if np.any(seed_dil[coords[:, 0], coords[:, 1]] > 0):
                local_keep[coords[:, 0], coords[:, 1]] = 1

        local_keep = local_keep.astype(np.uint8)
        gated[:, Lg:Rg] = np.maximum(gated[:, Lg:Rg], local_keep)

    return gated


def filter_final_blobs(final_mask, preserve_single_breakout=False):
    if not ENABLE_FINAL_BLOB_SIZE_FILTER:
        return final_mask

    H, W = final_mask.shape
    filtered = np.zeros_like(final_mask, dtype=np.uint8)
    kept = 0
    removed_small = 0
    removed_thin_vertical = 0
    removed_thin_horizontal = 0

    for region in regionprops(label(final_mask)):
        y0, x0, y1, x1 = region.bbox
        height = y1 - y0
        width = x1 - x0

        if region.area < MIN_FINAL_BLOB_AREA_PX:
            removed_small += 1
            continue

        if (
            not preserve_single_breakout
            and width <= MAX_THIN_VERTICAL_WIDTH_PX
            and height / (width + 1e-6) >= THIN_VERTICAL_MIN_ASPECT
        ):
            removed_thin_vertical += 1
            continue

        if height <= MAX_THIN_HORIZONTAL_HEIGHT_PX and width / (height + 1e-6) >= THIN_HORIZONTAL_MIN_ASPECT:
            removed_thin_horizontal += 1
            continue

        if ENABLE_LINE_LIKE_VERTICAL_STREAK_VETO and not preserve_single_breakout:
            local = final_mask[y0:y1, x0:x1]
            row_widths = []
            for yy in range(local.shape[0]):
                xs = np.where(local[yy] > 0)[0]
                if xs.size > 0:
                    row_widths.append(xs.max() - xs.min() + 1)

            if len(row_widths) > 0:
                row_widths = np.array(row_widths, dtype=float)
                median_row_width = float(np.median(row_widths))
                p90_row_width = float(np.percentile(row_widths, 90))
                row_width_std = float(np.std(row_widths))
                aspect_ratio = height / (width + 1e-6)

                line_like_vertical = (
                    height >= MIN_STREAK_HEIGHT_PX and
                    aspect_ratio >= MIN_STREAK_ASPECT_RATIO and
                    median_row_width <= MAX_STREAK_MEDIAN_ROW_WIDTH_PX and
                    p90_row_width <= MAX_STREAK_P90_ROW_WIDTH_PX and
                    row_width_std <= MAX_STREAK_ROW_WIDTH_STD_PX
                )

                if line_like_vertical:
                    removed_thin_vertical += 1
                    continue

        coords = region.coords
        filtered[coords[:, 0], coords[:, 1]] = 1
        kept += 1

    print(
        f"FINAL BLOB FILTER | kept={kept}, removed_small={removed_small}, "
        f"removed_thin_vertical={removed_thin_vertical}, "
        f"removed_thin_horizontal={removed_thin_horizontal}, min_area={MIN_FINAL_BLOB_AREA_PX}"
    )

    return filtered


def suppress_final_column_streaks(final_mask):
    if not ENABLE_FINAL_COLUMN_STREAK_SUPPRESSION:
        return final_mask

    H, W = final_mask.shape
    cleaned = final_mask.copy().astype(np.uint8)
    min_run = int(H * FINAL_COLUMN_STREAK_MIN_RUN_FRACTION)
    removed_cols = 0

    for x in range(W):
        col = final_mask[:, x]
        if np.sum(col) == 0:
            continue

        diff = np.diff(np.concatenate(([0], col, [0])))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]

        for s, e in zip(starts, ends):
            if e - s < min_run:
                continue

            x0 = max(0, x - FINAL_COLUMN_STREAK_LOCAL_HALF_WIDTH)
            x1 = min(W, x + FINAL_COLUMN_STREAK_LOCAL_HALF_WIDTH + 1)
            local = final_mask[s:e, x0:x1]
            local_widths = []
            for yy in range(local.shape[0]):
                xs = np.where(local[yy] > 0)[0]
                if xs.size > 0:
                    local_widths.append(xs.max() - xs.min() + 1)

            if len(local_widths) == 0:
                continue
            if np.median(local_widths) <= FINAL_COLUMN_STREAK_MAX_LOCAL_WIDTH_PX:
                cleaned[s:e, x] = 0
                removed_cols += 1

    if removed_cols > 0:
        print(f"FINAL COLUMN STREAK SUPPRESSION | removed_runs={removed_cols}")

    return cleaned


def local_azimuth_contrast_score(roi_image, L, R):
    H, W = roi_image.shape
    left0 = max(0, L - AZIMUTH_CONTRAST_MARGIN_PX)
    left1 = L
    right0 = R
    right1 = min(W, R + AZIMUTH_CONTRAST_MARGIN_PX)
    inside = roi_image[:, L:R]
    neighbors = []

    if left1 > left0:
        neighbors.append(roi_image[:, left0:left1])
    if right1 > right0:
        neighbors.append(roi_image[:, right0:right1])

    if inside.size == 0 or len(neighbors) == 0:
        return 0.0

    neighbor_values = np.concatenate([n.reshape(-1) for n in neighbors])
    return float(np.mean(neighbor_values) - np.mean(inside))


def print_quality(final_mask, roi_image, accepted_windows):
    for side_idx, (L, R) in enumerate(accepted_windows):
        mw = final_mask[:, L:R]
        widths = []

        for y in range(mw.shape[0]):
            xs = np.where(mw[y] > 0)[0]
            if xs.size > 0:
                widths.append(xs.max() - xs.min() + 1)

        if len(widths) == 0:
            print(f"POST-FILL QUALITY | side={side_idx}, LOW_EVIDENCE")
            continue

        widths = np.array(widths, dtype=float)
        contrast = local_azimuth_contrast_score(roi_image, L, R)

        print(
            f"POST-FILL QUALITY | side={side_idx}, mean_width={np.mean(widths):.2f}, "
            f"max_width={np.max(widths):.2f}, "
            f"wide_rows={np.mean(widths >= DIAG_MIN_MEAN_BREAKOUT_WIDTH_PX):.2f}, "
            f"contrast={contrast:.2f}"
        )



def pair_contrast_veto(final_mask, roi_image, accepted_windows):
    """
    Rejects accepted pairs whose azimuth windows are not actually darker
    than neighboring azimuths after final mask construction.
    """

    if not ENABLE_PAIR_CONTRAST_VETO:
        return False

    if accepted_windows is None or len(accepted_windows) != 2:
        return False

    contrasts = []

    for L, R in accepted_windows:
        contrast = local_azimuth_contrast_score(roi_image, L, R)
        contrasts.append(float(contrast))

    min_contrast = min(contrasts)
    mean_contrast = float(np.mean(contrasts))

    print(
        f"PAIR CONTRAST QC | "
        f"side_contrasts={contrasts}, "
        f"min={min_contrast:.2f}, "
        f"mean={mean_contrast:.2f}"
    )

    return (
        min_contrast < MIN_SIDE_CONTRAST
        or mean_contrast < MIN_MEAN_PAIR_CONTRAST
    )


def create_overlay(roi_rgb, final_mask, hotspot_boxes):
    overlay = roi_rgb.copy()
    blue = np.zeros_like(overlay)
    blue[..., 2] = 255
    alpha = 0.80
    idxs = final_mask == 1
    overlay[idxs] = ((1 - alpha) * overlay[idxs] + alpha * blue[idxs]).astype(np.uint8)

    for L, y0, R, y1 in hotspot_boxes:
        cv2.rectangle(overlay, (L, y0), (R, y1), (255, 0, 0), 2)

    return overlay


def save_qc_panel(out_path, roi_rgb, roi_proc, mask_clean, mask_detection, layer_rows, signal, threshold, overlay, accepted_windows, title):
    layer_diag = roi_rgb.copy()
    layer_diag[layer_rows, :, 0] = 255
    layer_diag[layer_rows, :, 1] = 0
    layer_diag[layer_rows, :, 2] = 0

    fig, axs = plt.subplots(1, 6, figsize=(34, 10))

    axs[0].imshow(roi_rgb)
    axs[0].set_title("Original ROI")
    axs[0].axis("off")

    axs[1].imshow(roi_proc, cmap="gray")
    axs[1].set_title("Processed ROI")
    axs[1].axis("off")

    axs[2].imshow(mask_clean, cmap="gray")
    axs[2].set_title("Raw dark mask")
    axs[2].axis("off")

    axs[3].imshow(layer_diag)
    axs[3].set_title("Layer rows excluded")
    axs[3].axis("off")

    axs[4].plot(np.linspace(0, 360, len(signal)), signal)
    axs[4].axhline(threshold, linestyle="--")
    if accepted_windows is not None:
        W = len(signal)
        for L, R in accepted_windows:
            axs[4].axvspan(360 * L / W, 360 * R / W, alpha=0.25)
    axs[4].set_title("Azimuth signal")
    axs[4].set_xlabel("Azimuth/degrees")

    axs[5].imshow(overlay)
    axs[5].set_title("Candidate overlay")
    axs[5].axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(out_path, dpi=250, bbox_inches="tight")
    plt.close()



def remove_layer_pixels_from_final_mask(final_mask, layer_rows):
    """
    Hard layer exclusion: pixels on rows identified as laterally continuous
    dark layers are not valid breakout pixels.
    """
    if not EXCLUDE_LAYER_PIXELS_FROM_FINAL_MASK:
        return final_mask

    cleaned = final_mask.copy().astype(np.uint8)
    removed = int(np.sum(cleaned[layer_rows, :] > 0))
    cleaned[layer_rows, :] = 0

    if removed > 0:
        print(f"FINAL LAYER PIXEL EXCLUSION | removed_pixels={removed}")

    return cleaned


def veto_layer_dominated_final_components(final_mask, layer_rows):
    """
    Remove connected components whose pixels or active rows overlap suppressed
    layer rows too much.
    """
    if not ENABLE_FINAL_LAYER_COMPONENT_VETO:
        return final_mask

    filtered = np.zeros_like(final_mask, dtype=np.uint8)
    removed = 0
    kept = 0

    for region in regionprops(label(final_mask)):
        coords = region.coords
        ys = coords[:, 0]

        layer_pixel_fraction = float(np.mean(layer_rows[ys]))

        component_rows = np.unique(ys)
        layer_row_fraction = float(np.mean(layer_rows[component_rows]))

        if (
            layer_pixel_fraction > MAX_FINAL_COMPONENT_LAYER_OVERLAP_FRACTION
            or layer_row_fraction > MAX_COMPONENT_LAYER_ROW_FRACTION
        ):
            removed += 1
            continue

        filtered[coords[:, 0], coords[:, 1]] = 1
        kept += 1

    if removed > 0:
        print(
            f"FINAL LAYER COMPONENT VETO | kept={kept}, removed={removed}, "
            f"max_pixel_overlap={MAX_FINAL_COMPONENT_LAYER_OVERLAP_FRACTION}, "
            f"max_row_overlap={MAX_COMPONENT_LAYER_ROW_FRACTION}"
        )

    return filtered


def breakout_is_layer_dominated(final_mask, layer_rows):
    """
    Final image/window-level layer-overlap veto.
    Returns True if final accepted breakout pixels are mostly supported by
    suppressed layer rows.
    """
    if not ENABLE_WINDOW_LAYER_OVERLAP_VETO:
        return False

    if np.sum(final_mask) == 0:
        return True

    mask_pixels = final_mask > 0

    layer_pixel_fraction = (
        np.sum(mask_pixels & layer_rows[:, None]) /
        (np.sum(mask_pixels) + 1e-6)
    )

    active_rows = np.any(mask_pixels, axis=1)

    if np.sum(active_rows) == 0:
        return True

    layer_row_fraction = (
        np.sum(active_rows & layer_rows) /
        (np.sum(active_rows) + 1e-6)
    )

    print(
        f"LAYER OVERLAP QC | "
        f"layer_pixel_fraction={layer_pixel_fraction:.2f}, "
        f"layer_row_fraction={layer_row_fraction:.2f}"
    )

    return (
        layer_pixel_fraction > MAX_WINDOW_LAYER_PIXEL_FRACTION
        or layer_row_fraction > MAX_WINDOW_LAYER_ROW_FRACTION
    )


def pair_window_support_metrics(mask, L, R, margin_px=0):
    """
    Measures support of a candidate lobe within an azimuth window.

    Returns:
      area: number of mask pixels
      top/bottom: active row span
      active_rows: number of rows containing at least one mask pixel
    """
    H, W = mask.shape

    Lm = max(0, int(L) - int(margin_px))
    Rm = min(W, int(R) + int(margin_px))

    local = mask[:, Lm:Rm] > 0
    area = int(np.sum(local))

    if area == 0:
        return {
            "area": 0,
            "top": None,
            "bottom": None,
            "active_rows": 0
        }

    rows = np.where(np.any(local, axis=1))[0]

    return {
        "area": area,
        "top": int(rows.min()),
        "bottom": int(rows.max()) + 1,
        "active_rows": int(rows.size)
    }


def validate_pair_quality(mask_for_qc, accepted_windows):
    """
    Rejects false positives where one side dominates or the two lobes do not
    share enough vertical/depth support.

    This is a geometry gate for breakout-like azimuthal symmetry.
    """
    if not ENABLE_PAIR_QUALITY_GATE:
        return True

    if accepted_windows is None or len(accepted_windows) != 2:
        return False

    (L1, R1), (L2, R2) = accepted_windows

    m1 = pair_window_support_metrics(
        mask_for_qc,
        L1,
        R1,
        margin_px=PAIR_QC_AZIMUTH_MARGIN_PX
    )

    m2 = pair_window_support_metrics(
        mask_for_qc,
        L2,
        R2,
        margin_px=PAIR_QC_AZIMUTH_MARGIN_PX
    )

    area1 = m1["area"]
    area2 = m2["area"]

    if area1 == 0 or area2 == 0:
        print(
            f"PAIR QC | rejected empty side | "
            f"area1={area1}, area2={area2}"
        )
        return False

    area_balance = min(area1, area2) / (max(area1, area2) + 1e-6)

    top1, bot1 = m1["top"], m1["bottom"]
    top2, bot2 = m2["top"], m2["bottom"]

    overlap = max(0, min(bot1, bot2) - max(top1, top2))
    union = max(bot1, bot2) - min(top1, top2)
    vertical_overlap = overlap / (union + 1e-6)

    print(
        f"PAIR QC | "
        f"area1={area1}, area2={area2}, "
        f"area_balance={area_balance:.2f}, "
        f"vertical_overlap={vertical_overlap:.2f}"
    )

    if area_balance < MIN_PAIR_AREA_BALANCE:
        print("PAIR REJECTED: poor lobe area balance")
        return False

    if vertical_overlap < MIN_PAIR_VERTICAL_OVERLAP:
        print("PAIR REJECTED: poor vertical overlap between lobes")
        return False

    return True


def process_one_image(file, mode, output_folder, confirmed_direction, previous_image_positive, segmentation_method):
    """
    Process one image and return:
      confirmed_direction
      result_row dictionary for automatic QC table

    code_breakout_detected is True only when the full pipeline leaves
    non-zero pixels in final_mask after final gating/blob filtering.
    """

    result_row = {
        "image_file": file,
        "mode": mode,
        "segmentation_method": segmentation_method,
        "code_breakout_detected": False,
        "pair_candidate_found": False,
        "single_breakout_found": False,
        "breakout_mode": "none",
        "pair_quality_passed": False,
        "candidate_count": 0,
        "best_pair_score": np.nan,
        "final_mask_pixels": 0,
        "final_mask_area_fraction": 0.0,
        "azimuth_1_deg": np.nan,
        "azimuth_2_deg": np.nan,
        "azimuth_1_width_deg": np.nan,
        "azimuth_2_width_deg": np.nan,
        "pair_separation_deg": np.nan,
        "pair_quality_area_balance": np.nan,
        "pair_quality_vertical_overlap": np.nan,
        "pair_strength_balance": np.nan,
        "pair_energy_balance": np.nan,
        "pair_width_balance": np.nan,
        "competing_peak_to_weak": np.nan,
        "competing_energy_to_weak": np.nan,
        "single_azimuth_deg": np.nan,
        "single_azimuth_width_deg": np.nan,
        "single_azimuth_error_deg": np.nan,
        "single_dominance": np.nan,
        "single_peak_bg_ratio": np.nan,
        "single_isolation": np.nan,
        "single_score": np.nan,
        "direction_memory_deg": np.nan,
        "prior_direction_deg": float(confirmed_direction) if confirmed_direction is not None else np.nan,
        "previous_image_positive": bool(previous_image_positive),
        "continuity_rescue_used": False,
        "continuity_prior_direction_deg": np.nan,
        "continuity_drift_deg": np.nan,
        "continuity_peak_bg_ratio": np.nan,
        "continuity_score": np.nan,
        "status_message": "not_processed",
        "visual_confirm": "",
        "failure_reason": "",
        "comment": ""
    }

    path = os.path.join(INPUT_FOLDER, file)
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        print("Could not read image. Skipping.")
        result_row["status_message"] = "could_not_read_image"
        return confirmed_direction, previous_image_positive, result_row

    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    if USE_FULL_IMAGE_AS_ROI:
        roi = gray
        roi_rgb = img
    else:
        # Fallback if you later want to manually define coordinates again.
        roi = gray[y_roi:y_roi + h_roi, x_roi:x_roi + w_roi]
        roi_rgb = img[y_roi:y_roi + h_roi, x_roi:x_roi + w_roi]

    H, W = roi.shape

    if H == 0 or W == 0:
        print("Invalid ROI. Skipping.")
        result_row["status_message"] = "invalid_roi"
        return confirmed_direction, previous_image_positive, result_row

    window_height, window_step = get_manual_windows(H)
    roi_eq, roi_proc = preprocess_roi(roi, mode)

    mask_raw, labels, means = segment_dark_pixels(
        roi_proc,
        method=segmentation_method
    )

    mask_clean = remove_small_objects(mask_raw.astype(bool), min_size=MIN_OBJECT_SIZE)
    mask_clean = remove_small_holes(mask_clean, area_threshold=MAX_HOLE_SIZE)
    mask_clean = mask_clean.astype(np.uint8)

    mask_no_layers, layer_rows = suppress_laterally_continuous_layers(mask_clean)

    # Candidate detection mask: suppress beds and narrow vertical streaks.
    mask_detection_input = remove_vertical_line_artifacts_for_detection(mask_no_layers)
    weighted_mask, mask_detection = build_verticality_weighted_mask(mask_detection_input)

    # Final labeling mask starts from cleaned dark mask, then hard-excludes layer pixels.
    _, mask_labeling = build_verticality_weighted_mask(mask_clean)
    if EXCLUDE_LAYER_PIXELS_FROM_FINAL_MASK:
        mask_labeling[layer_rows, :] = 0

    signal = compute_azimuth_signal(mask_detection)
    signal = reject_artifact_like_azimuth_columns(mask_detection, signal)

    regions, threshold = find_active_regions(signal, W)
    candidates = build_candidates(regions, signal, weighted_mask, W)
    candidates = merge_wraparound_candidates(candidates, W)
    result_row["candidate_count"] = int(len(candidates))
    print(f"Candidates found: {len(candidates)}")

    best_pair, best_score = search_best_pair(candidates, signal, confirmed_direction)
    result_row["best_pair_score"] = float(best_score) if best_score is not None else np.nan

    final_mask = np.zeros_like(mask_labeling, dtype=np.uint8)
    hotspot_boxes = []
    accepted_windows = None

    if best_pair is not None:
        result_row["pair_candidate_found"] = True

        c1, c2 = best_pair
        accepted_windows = [(c1["L"], c1["R"]), (c2["L"], c2["R"])]

        # Pair quality gate is evaluated on mask_labeling before growth/fill.
        # This rejects unilateral clutter that lacks a conjugate lobe.
        pair_quality_ok = validate_pair_quality(mask_labeling, accepted_windows)
        result_row["pair_quality_passed"] = bool(pair_quality_ok)

        if not pair_quality_ok:
            print("NO VALID BREAKOUT: pair failed quality gate")
            result_row["status_message"] = "rejected_pair_quality"
            accepted_windows = None

        else:
            proposed_direction = (c1["phi"] + c2["phi"]) / 2

            result_row["azimuth_1_deg"] = float(c1["phi"])
            result_row["azimuth_2_deg"] = float(c2["phi"])
            result_row["azimuth_1_width_deg"] = float(c1["Wdeg"])
            result_row["azimuth_2_width_deg"] = float(c2["Wdeg"])
            result_row["pair_separation_deg"] = float(circ_distance(c1["phi"], c2["phi"]))
            result_row["direction_memory_deg"] = float(proposed_direction)

            result_row["valley_depth"] = float(c1.get("valley_depth", np.nan))
            result_row["valley_noise"] = float(c1.get("valley_noise", np.nan))
            result_row["peak_bg_ratio"] = float(c1.get("peak_bg_ratio", np.nan))
            result_row["pair_isolation"] = float(c1.get("pair_isolation", np.nan))
            result_row["valley_width_frac"] = float(c1.get("valley_width_frac", np.nan))
            result_row["plateau_mean_ratio"] = float(c1.get("plateau_mean_ratio", np.nan))
            result_row["weak_effective_peak"] = float(c1.get("weak_effective_peak", np.nan))
            result_row["pair_strength_balance"] = float(c1.get("pair_strength_balance", np.nan))
            result_row["pair_energy_balance"] = float(c1.get("pair_energy_balance", np.nan))
            result_row["pair_width_balance"] = float(c1.get("pair_width_balance", np.nan))
            result_row["competing_peak_to_weak"] = float(c1.get("competing_peak_to_weak", np.nan))
            result_row["competing_energy_to_weak"] = float(c1.get("competing_energy_to_weak", np.nan))

            print("ACCEPTED BREAKOUT CANDIDATE PAIR")
            print(f"  Azimuth 1: {c1['phi']:.1f} deg, width: {c1['Wdeg']:.1f} deg")
            print(f"  Azimuth 2: {c2['phi']:.1f} deg, width: {c2['Wdeg']:.1f} deg")
            print(f"  Pair direction memory: {proposed_direction:.1f} deg")

            final_mask, hotspot_boxes = validate_vertical_windows(
                mask_labeling,
                accepted_windows,
                window_height,
                window_step
            )

            final_mask = expand_dark_fill_in_accepted_windows(
                final_mask,
                roi_proc,
                accepted_windows,
                layer_rows
            )

            # Hard rule: layer pixels are not valid breakout pixels.
            final_mask = remove_layer_pixels_from_final_mask(final_mask, layer_rows)

            final_mask = apply_final_darkest_pixel_gate(
                final_mask,
                roi_proc,
                accepted_windows
            )

            # Apply layer exclusion again after darkest-pixel expansion.
            final_mask = remove_layer_pixels_from_final_mask(final_mask, layer_rows)
            final_mask = veto_layer_dominated_final_components(final_mask, layer_rows)

            final_mask = filter_final_blobs(final_mask)
            labels = label(final_mask)
            n_final_blobs = labels.max()

            print(
                f"FINAL MASK QC | "
                f"n_blobs={n_final_blobs}"
            )

            if (
                IMAGE_REGIME == "cave_rich"
                and ENABLE_CAVE_RICH_POST_FILL_FRAGMENT_VETO
                and n_final_blobs > MAX_CAVE_RICH_FINAL_BLOBS
            ):
                print(
                    f"REJECTED: cave-rich fragmented overfill | "
                    f"n_blobs={n_final_blobs}, "
                    f"max={MAX_CAVE_RICH_FINAL_BLOBS}"
                )
                final_mask[:, :] = 0
                result_row["status_message"] = "rejected_cave_rich_fragmented_overfill"

            final_mask = suppress_final_column_streaks(final_mask)

            if ENABLE_PAIR_CONTRAST_VETO:
                if pair_contrast_veto(final_mask, roi_proc, accepted_windows):
                    print("REJECTED: insufficient breakout-side contrast")
                    final_mask[:, :] = 0
                    result_row["status_message"] = "rejected_low_pair_contrast"
            if ENABLE_WINDOW_LAYER_OVERLAP_VETO:
                if breakout_is_layer_dominated(final_mask, layer_rows):
                    print("REJECTED: final mask is dominated by suppressed layer rows")
                    final_mask[:, :] = 0
                    result_row["status_message"] = "rejected_layer_dominated"

            if ENABLE_POST_FILL_QUALITY_REPORT and accepted_windows is not None:
                print_quality(final_mask, roi_proc, accepted_windows)

            final_pixels = int(np.sum(final_mask > 0))
            result_row["final_mask_pixels"] = final_pixels
            result_row["final_mask_area_fraction"] = float(final_pixels / (H * W + 1e-6))
            result_row["code_breakout_detected"] = bool(final_pixels > 0)
            if final_pixels > 0:
                result_row["breakout_mode"] = "pair"
                confirmed_direction = float(proposed_direction)

            if result_row["status_message"] not in ["rejected_layer_dominated", "rejected_low_pair_contrast"]:
                result_row["status_message"] = (
                    "breakout_detected_pair"
                    if final_pixels > 0
                    else "pair_found_but_final_mask_empty"
                )

    else:
        print("NO VALID BREAKOUT CANDIDATE PAIR FOUND")
        result_row["status_message"] = "no_valid_pair_found"


    # =====================================================
    # SINGLE 180-DEGREE BREAKOUT FALLBACK
    # =====================================================
    # This runs only if the pair path did not leave a valid final mask.
    # It is intentionally restricted to a dominant lobe around 180 degrees.
    if (
        ENABLE_SINGLE_BREAKOUT_MODE
        and not result_row["code_breakout_detected"]
    ):

        best_single, single_score, single_metrics = search_best_single_breakout(
            candidates,
            signal
        )

        single_mode_label = "single_180"

        if best_single is None:
            best_single, single_score, single_metrics = search_continuity_single_rescue(
                candidates,
                signal,
                confirmed_direction,
                previous_image_positive
            )
            if best_single is not None:
                single_mode_label = "continuity_single"

        if best_single is not None:
            print(
                f"ACCEPTED SINGLE BREAKOUT FALLBACK | "
                f"phi={best_single['phi']:.1f} deg, "
                f"width={best_single['Wdeg']:.1f} deg, "
                f"score={single_score:.5f}"
            )

            accepted_windows = [(best_single["L"], best_single["R"])]

            result_row["single_breakout_found"] = True
            result_row["breakout_mode"] = single_mode_label
            result_row["single_azimuth_deg"] = float(best_single["phi"])
            result_row["single_azimuth_width_deg"] = float(best_single["Wdeg"])
            result_row["single_score"] = float(single_score)

            for k, v in single_metrics.items():
                result_row[k] = float(v)

            final_mask, hotspot_boxes = validate_vertical_windows(
                mask_labeling,
                accepted_windows,
                window_height,
                window_step
            )

            final_mask = expand_dark_fill_in_accepted_windows(
                final_mask,
                roi_proc,
                accepted_windows,
                layer_rows
            )

            final_mask = remove_layer_pixels_from_final_mask(final_mask, layer_rows)

            final_mask = apply_final_darkest_pixel_gate(
                final_mask,
                roi_proc,
                accepted_windows
            )

            final_mask = remove_layer_pixels_from_final_mask(final_mask, layer_rows)
            final_mask = veto_layer_dominated_final_components(final_mask, layer_rows)

            final_mask = filter_final_blobs(
                final_mask,
                preserve_single_breakout=PRESERVE_SINGLE_BREAKOUT_VERTICAL_BLOB
            )
            final_mask = suppress_final_column_streaks(final_mask)

            if ENABLE_WINDOW_LAYER_OVERLAP_VETO:
                if breakout_is_layer_dominated(final_mask, layer_rows):
                    print("REJECTED: single final mask is dominated by suppressed layer rows")
                    final_mask[:, :] = 0
                    result_row["status_message"] = "rejected_single_layer_dominated"

            if ENABLE_POST_FILL_QUALITY_REPORT and accepted_windows is not None:
                print_quality(final_mask, roi_proc, accepted_windows)

            final_pixels = int(np.sum(final_mask > 0))
            result_row["final_mask_pixels"] = final_pixels
            result_row["final_mask_area_fraction"] = float(final_pixels / (H * W + 1e-6))
            result_row["code_breakout_detected"] = bool(final_pixels > 0)

            if final_pixels > 0:
                confirmed_direction = float(best_single["phi"])

            result_row["status_message"] = (
                f"breakout_detected_{single_mode_label}"
                if final_pixels > 0
                else f"{single_mode_label}_found_but_final_mask_empty"
            )

        else:
            if result_row["status_message"] in ["not_processed", ""]:
                result_row["status_message"] = "no_valid_pair_or_single_found"
            elif result_row["status_message"] == "no_valid_pair_found":
                result_row["status_message"] = "no_valid_pair_or_single_found"

    overlay = create_overlay(roi_rgb, final_mask, hotspot_boxes)

    base = os.path.splitext(file)[0]
    qc_path = os.path.join(output_folder, f"{base}_QC_{mode}.jpg")
    mask_path = os.path.join(output_folder, f"{base}_mask_{mode}.png")
    overlay_path = os.path.join(output_folder, f"{base}_overlay_{mode}.png")

    save_qc_panel(
        qc_path,
        roi_rgb,
        roi_proc,
        mask_clean,
        mask_detection,
        layer_rows,
        signal,
        threshold,
        overlay,
        accepted_windows,
        title=f"{file} | {mode}"
    )

    cv2.imwrite(mask_path, (final_mask * 255).astype(np.uint8))
    cv2.imwrite(overlay_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    return confirmed_direction, previous_image_positive, result_row


# =====================================================
# STACK RESULT IMAGES
# =====================================================

def natural_sort_key(path):
    """
    Natural sorting for depth/file-number ordered image names.
    Example: image_2 comes before image_10.
    """
    base = os.path.basename(path)

    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"([0-9]+)", base)
    ]


def stack_result_images(
    folder,
    image_type="overlay",
    save_png=True,
    open_after_save=False
):
    """
    Stacks saved result images vertically after the pipeline finishes.

    image_type:
      "overlay" -> stacks *_overlay_*.png/jpg/jpeg
      "QC"      -> stacks *_QC_*.jpg/png/jpeg
    """

    if image_type == "overlay":
        patterns = [
            "*overlay*.png",
            "*overlay*.jpg",
            "*overlay*.jpeg"
        ]

    elif image_type == "QC":
        patterns = [
            "*QC*.jpg",
            "*QC*.png",
            "*QC*.jpeg"
        ]

    else:
        raise ValueError("image_type must be 'overlay' or 'QC'")

    stack_files = []

    for pat in patterns:
        stack_files.extend(
            glob.glob(
                os.path.join(
                    folder,
                    pat
                )
            )
        )

    stack_files = sorted(
        stack_files,
        key=natural_sort_key
    )

    print(f"\nSTACKING | type={image_type} | found={len(stack_files)} images")

    if len(stack_files) == 0:
        print("No result images found to stack.")
        return None

    imgs = []

    for p in stack_files:
        img = cv2.imread(p)

        if img is None:
            print(f"WARNING: could not read {p}")
            continue

        imgs.append(img)

    if len(imgs) == 0:
        print("No readable images found.")
        return None

    min_w = min(
        img.shape[1]
        for img in imgs
    )

    resized = []

    for img in imgs:
        h, w = img.shape[:2]

        if w != min_w:
            scale = min_w / w

            img = cv2.resize(
                img,
                (
                    min_w,
                    int(h * scale)
                )
            )

        resized.append(img)

    ext = "png" if save_png else "jpg"
    max_stack_height_px = 60000

    total_h = int(sum(img.shape[0] for img in resized))

    saved_paths = []

    if total_h <= max_stack_height_px:
        final_stack = np.vstack(resized)
        save_path = os.path.join(folder, f"STACKED_{image_type}.{ext}")
        ok = cv2.imwrite(save_path, final_stack)
        print("STACKING | saved OK:", ok)
        print(save_path)
        if ok:
            saved_paths.append(save_path)
    else:
        print(f"STACKING | total height {total_h}px is large; saving in chunks")
        chunk = []
        chunk_h = 0
        chunk_idx = 1

        for img in resized:
            if chunk and chunk_h + img.shape[0] > max_stack_height_px:
                stack_chunk = np.vstack(chunk)
                save_path = os.path.join(folder, f"STACKED_{image_type}_part{chunk_idx:02d}.{ext}")
                ok = cv2.imwrite(save_path, stack_chunk)
                print("STACKING | saved chunk OK:", ok, save_path)
                if ok:
                    saved_paths.append(save_path)
                chunk = []
                chunk_h = 0
                chunk_idx += 1

            chunk.append(img)
            chunk_h += img.shape[0]

        if chunk:
            stack_chunk = np.vstack(chunk)
            save_path = os.path.join(folder, f"STACKED_{image_type}_part{chunk_idx:02d}.{ext}")
            ok = cv2.imwrite(save_path, stack_chunk)
            print("STACKING | saved chunk OK:", ok, save_path)
            if ok:
                saved_paths.append(save_path)

    manifest_path = os.path.join(folder, f"STACKED_{image_type}_manifest.txt")
    with open(manifest_path, "w", encoding="utf-8") as f:
        for p in saved_paths:
            f.write(p + "\n")
    print("STACKING | manifest:", manifest_path)

    if saved_paths and open_after_save:
        try:
            os.startfile(saved_paths[0])
        except Exception as exc:
            print(f"Could not auto-open stacked image: {exc}")

    return saved_paths[0] if len(saved_paths) == 1 else saved_paths


# =====================================================
# STACKING SETTINGS
# =====================================================

STACK_RESULTS_AT_END = True
STACK_IMAGE_TYPE = "overlay"   # "overlay" or "QC"
STACK_OPEN_AFTER_SAVE = False  # set True if you want Windows to open the PNG automatically

apply_image_regime_preset()

# =====================================================
# MAIN LOOP + KMEANS + AUTOMATIC DETECTION TABLE
# =====================================================

qc_rows = []

for segmentation_method in SEGMENTATION_MODES:

    mode = PREPROCESS_MODE

    print("\n################################################")
    print(f"PREPROCESS: {mode}")
    print(f"SEGMENTATION: {segmentation_method}")
    print("################################################")

    output_folder = os.path.join(
        OUTPUT_ROOT,
        f"{mode}_{segmentation_method}"
    )
    os.makedirs(output_folder, exist_ok=True)

    confirmed_direction = None
    previous_image_positive = False

    for idx, file in enumerate(files):

        print("\n================================================")
        print(f"{segmentation_method} | IMAGE {idx + 1}/{len(files)}")
        print(file)
        print("================================================")

        confirmed_direction, previous_image_positive, result_row = process_one_image(
            file=file,
            mode=mode,
            output_folder=output_folder,
            confirmed_direction=confirmed_direction,
            previous_image_positive=previous_image_positive,
            segmentation_method=segmentation_method
        )

        previous_image_positive = bool(result_row.get("code_breakout_detected", False))

        qc_rows.append(result_row)


# =====================================================
# BUILD AND SAVE AUTOMATIC QC TABLES
# =====================================================

qc = pd.DataFrame(qc_rows)

qc_folder = os.path.join(OUTPUT_ROOT, "_automatic_qc_tables")
os.makedirs(qc_folder, exist_ok=True)

qc_csv_path = os.path.join(qc_folder, "automatic_breakout_detection_qc.csv")
qc.to_csv(qc_csv_path, index=False)

summary = (
    qc.groupby(["mode", "segmentation_method"], dropna=False)
      .agg(
          total_images=("image_file", "count"),
          code_breakout_detected_count=("code_breakout_detected", "sum"),
          pair_candidate_found_count=("pair_candidate_found", "sum"),
          mean_final_mask_pixels=("final_mask_pixels", "mean"),
          mean_candidate_count=("candidate_count", "mean")
      )
      .reset_index()
)

summary["code_breakout_detected_percent"] = (
    100 * summary["code_breakout_detected_count"] / summary["total_images"]
).round(1)

summary["pair_candidate_found_percent"] = (
    100 * summary["pair_candidate_found_count"] / summary["total_images"]
).round(1)

summary_csv_path = os.path.join(qc_folder, "automatic_breakout_detection_summary.csv")
summary.to_csv(summary_csv_path, index=False)

print("\nDONE")
print("Automatic detection QC table saved to:")
print(qc_csv_path)
print("Automatic detection summary saved to:")
print(summary_csv_path)

print("\nAutomatic summary:")
display(summary)


# =====================================================
# VISUAL QC TABLE: ONE ROW PER IMAGE
# =====================================================

visual_rows = []

for segmentation_method in SEGMENTATION_MODES:

    sub_qc = qc[qc["segmentation_method"] == segmentation_method]

    for image_file in sorted(sub_qc["image_file"].unique(), key=extract_depth_key):

        sub = sub_qc[sub_qc["image_file"] == image_file]

        if len(sub) == 0:
            row = {
                "image_file": image_file,
                "segmentation_method": segmentation_method,
                "bilateral_only_code_detected": "",
                "pair_candidate_found": "",
                "pair_quality_passed": "",
                "candidate_count": "",
                "final_mask_pixels": "",
                "final_mask_area_fraction": "",
                "azimuth_1_deg": "",
                "azimuth_2_deg": "",
                "pair_separation_deg": "",
                "valley_depth": "",
                "valley_noise": "",
                "peak_bg_ratio": "",
                "pair_isolation": "",
                "valley_width_frac": "",
                "plateau_mean_ratio": "",
                "weak_effective_peak": "",
                "status_message": "",
                "manual": "",
                "quality_class": "",
            "reason": "",
                "comment": ""
            }
        else:
            s = sub.iloc[0]
            row = {
                "image_file": image_file,
                "segmentation_method": segmentation_method,
                "bilateral_only_code_detected": bool(s["code_breakout_detected"]),
                "pair_candidate_found": bool(s["pair_candidate_found"]),
                "pair_quality_passed": bool(s.get("pair_quality_passed", False)),
                "candidate_count": s["candidate_count"],
                "final_mask_pixels": s["final_mask_pixels"],
                "final_mask_area_fraction": s["final_mask_area_fraction"],
                "azimuth_1_deg": s["azimuth_1_deg"],
                "azimuth_2_deg": s["azimuth_2_deg"],
                "pair_separation_deg": s["pair_separation_deg"],
                "valley_depth": s.get("valley_depth", np.nan),
                "valley_noise": s.get("valley_noise", np.nan),
                "peak_bg_ratio": s.get("peak_bg_ratio", np.nan),
                "pair_isolation": s.get("pair_isolation", np.nan),
                "valley_width_frac": s.get("valley_width_frac", np.nan),
                "plateau_mean_ratio": s.get("plateau_mean_ratio", np.nan),
                "weak_effective_peak": s.get("weak_effective_peak", np.nan),
                "status_message": s["status_message"],
                "manual": "",
                "quality_class": "",
            "reason": "",
                "comment": ""
            }

        visual_rows.append(row)

visual_qc = pd.DataFrame(visual_rows)

visual_qc_path = os.path.join(qc_folder, "visual_qc_kmeans_only_template.csv")

try:
    visual_qc.to_csv(visual_qc_path, index=False)
except PermissionError:
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    visual_qc_path = os.path.join(qc_folder, f"visual_qc_kmeans_som_template_{timestamp}.csv")
    visual_qc.to_csv(visual_qc_path, index=False)

print("\nVisual QC table created:")
print(visual_qc_path)
display(visual_qc)


# =====================================================
# STATS BLOCK + TWO-SLOT DROPDOWN VISUAL QC FORM
# =====================================================
# QC slots:
#   quality_class = good / fair / poor
#   reason        = none / wrong_azimuth / oversegmented / undersegmented /
#                   layer_leakage / artifact_leakage / missed_breakout /
#                   wide_breakout_clipped
# =====================================================

def _to_bool(series):
    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .replace({
            "verdadeiro": "true",
            "falso": "false",
            "sim": "true",
            "não": "false",
            "nao": "false",
            "x": "true"
        })
        .isin(["true", "1", "yes", "valid", "v", "t"])
    )


def summarize_visual_qc(
    visual_qc_csv_path=visual_qc_path,
    output_folder=qc_folder
):
    """
    Computes:
      1. presence/absence stats: code vs manual
      2. quality_class distribution
      3. reason distribution
    """

    qcv = pd.read_csv(visual_qc_csv_path)

    qcv.columns = (
        qcv.columns
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Backward compatibility
    if "quality_class" not in qcv.columns and "visual_match" in qcv.columns:
        qcv["quality_class"] = qcv["visual_match"]

    if "reason" not in qcv.columns:
        if "failure_modes" in qcv.columns:
            qcv["reason"] = qcv["failure_modes"]
        elif "failure_reason" in qcv.columns:
            qcv["reason"] = qcv["failure_reason"]
        else:
            qcv["reason"] = ""

    required = [
        "image_file",
        "bilateral_only_code_detected",
        "manual",
        "quality_class",
        "reason"
    ]

    missing = [c for c in required if c not in qcv.columns]
    if missing:
        raise KeyError(f"Missing required columns in visual QC CSV: {missing}")

    code = _to_bool(qcv["bilateral_only_code_detected"])
    manual = _to_bool(qcv["manual"])

    quality = (
        qcv["quality_class"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    reason = (
        qcv["reason"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    usable = quality.isin(["good", "fair"])

    TP = int(np.sum(code & manual))
    FP = int(np.sum(code & (~manual)))
    FN = int(np.sum((~code) & manual))
    TN = int(np.sum((~code) & (~manual)))

    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    accuracy = (TP + TN) / len(qcv) if len(qcv) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else np.nan
    )

    quality_counts = (
        quality
        .replace("", np.nan)
        .dropna()
        .value_counts()
        .rename_axis("quality_class")
        .reset_index(name="count")
    )

    reason_counts = (
        reason
        .replace("", np.nan)
        .replace("nan", np.nan)
        .dropna()
        .value_counts()
        .rename_axis("reason")
        .reset_index(name="count")
    )

    stats = pd.DataFrame([{
        "mode": "bilateral_only_kmeans",
        "TP_presence": TP,
        "FP_presence": FP,
        "FN_presence": FN,
        "TN_presence": TN,
        "presence_precision": round(precision, 3) if not np.isnan(precision) else np.nan,
        "presence_recall": round(recall, 3) if not np.isnan(recall) else np.nan,
        "presence_accuracy": round(accuracy, 3) if not np.isnan(accuracy) else np.nan,
        "presence_f1": round(f1, 3) if not np.isnan(f1) else np.nan,
        "good": int(np.sum(quality == "good")),
        "fair": int(np.sum(quality == "fair")),
        "poor": int(np.sum(quality == "poor")),
        "usable_annotations": int(np.sum(usable)),
        "usable_percent": round(100 * usable.mean(), 1) if len(qcv) > 0 else np.nan,
        "n_images": len(qcv)
    }])

    stats_path = os.path.join(output_folder, "manual_visual_stats_kmeans_only.csv")
    detail_path = os.path.join(output_folder, "manual_visual_detail_kmeans_only.csv")
    quality_path = os.path.join(output_folder, "quality_class_counts_kmeans_only.csv")
    reason_path = os.path.join(output_folder, "reason_counts_kmeans_only.csv")

    stats.to_csv(stats_path, index=False)
    qcv.to_csv(detail_path, index=False)
    quality_counts.to_csv(quality_path, index=False)
    reason_counts.to_csv(reason_path, index=False)

    print("\nManual + visual QC stats:")
    display(stats)

    print("\nQuality class counts:")
    display(quality_counts)

    print("\nReason counts:")
    display(reason_counts)

    print("\nSaved:")
    print(stats_path)
    print(detail_path)
    print(quality_path)
    print(reason_path)

    return stats, quality_counts, reason_counts, qcv

def launch_visual_qc_form(
    qc_csv_path=visual_qc_path,
    overlay_folder=None
):
    """
    Dropdown visual QC form.

    Two annotation slots:
      quality_class: good / fair / poor
      reason: none / wrong_azimuth / oversegmented / undersegmented / etc.

    Saves directly to the QC CSV.
    """

    import os
    import pandas as pd
    from IPython.display import display, Image, clear_output

    try:
        import ipywidgets as widgets
    except ModuleNotFoundError:
        print("ipywidgets is not installed.")
        print("Install with: %pip install ipywidgets")
        print("Then restart the kernel and run launch_visual_qc_form() again.")
        return

    qcv = pd.read_csv(qc_csv_path)

    QC_DISPLAY_HEIGHT = 300

    # Ensure string-editable columns. This avoids pandas float64 assignment errors.
    for col in ["manual", "quality_class", "reason", "comment"]:

        if col not in qcv.columns:
            qcv[col] = ""

        qcv[col] = (
            qcv[col]
            .fillna("")
            .astype(str)
            .replace("nan", "")
        )

    # Backward compatibility with previous columns
    if "failure_modes" in qcv.columns:
        empty_reason = qcv["reason"].astype(str).str.strip().eq("")
        qcv.loc[empty_reason, "reason"] = (
            qcv.loc[empty_reason, "failure_modes"]
            .fillna("")
            .astype(str)
            .replace("nan", "")
        )

    if "visual_match" in qcv.columns:
        empty_quality = qcv["quality_class"].astype(str).str.strip().eq("")
        qcv.loc[empty_quality, "quality_class"] = (
            qcv.loc[empty_quality, "visual_match"]
            .fillna("")
            .astype(str)
            .replace("nan", "")
        )

    state = {"idx": 0}

    manual_dd = widgets.Dropdown(
        options=["", "True", "False"],
        description="Manual:"
    )

    quality_dd = widgets.Dropdown(
        options=["", "good", "fair", "poor"],
        description="Quality:"
    )

    reason_dd = widgets.Dropdown(
        options=[
            "",
            "none",
            "wrong_azimuth",
            "oversegmented",
            "undersegmented",
            "layer_leakage",
            "artifact_leakage",
            "missed_breakout",
            "wide_breakout_clipped",
            "low_pair_contrast",
            "broad_plateau_cave_like",
            "cave_rich_multimodal_signal",
            "weak_valley_or_multimodal_signal",
            "same_plateau_multipeak_false_positive"
        ],
        description="Reason:"
    )

    comment_txt = widgets.Textarea(
        description="Comment:",
        layout=widgets.Layout(width="720px", height="80px")
    )

    prev_btn = widgets.Button(description="Previous")
    next_btn = widgets.Button(description="Save + Next", button_style="success")
    save_btn = widgets.Button(description="Save", button_style="info")
    stats_btn = widgets.Button(description="Save + Stats", button_style="warning")

    out = widgets.Output()

    def _clean(x):
        if pd.isna(x):
            return ""
        val = str(x)
        if val.lower() == "nan":
            return ""
        return val

    def get_overlay_path(row):
        image_file = row["image_file"]
        base = os.path.splitext(image_file)[0]

        candidate_folders = []

        if overlay_folder is not None:
            candidate_folders.append(overlay_folder)

        candidate_folders.extend([
            os.path.join(OUTPUT_ROOT, f"{PREPROCESS_MODE}_kmeans"),
            os.path.join(OUTPUT_ROOT, PREPROCESS_MODE),
        ])

        for folder in candidate_folders:
            p = os.path.join(folder, f"{base}_overlay_{PREPROCESS_MODE}.png")
            if os.path.exists(p):
                return p

        return None

    def render():
        with out:
            clear_output(wait=True)

            row = qcv.iloc[state["idx"]]

            print(f"{state['idx'] + 1}/{len(qcv)} | {row['image_file']}")
            print(f"Code detected: {row.get('bilateral_only_code_detected', '')}")
            print(f"Pair found: {row.get('pair_candidate_found', '')}")
            print(f"Pair QC passed: {row.get('pair_quality_passed', '')}")
            print(f"Status: {row.get('status_message', '')}")
            print(f"Azimuths: {row.get('azimuth_1_deg', '')} / {row.get('azimuth_2_deg', '')}")
            print(f"Pair separation: {row.get('pair_separation_deg', '')}")

            overlay_path = get_overlay_path(row)

            if overlay_path and os.path.exists(overlay_path):

                img = cv2.imread(overlay_path)

                if img is None:
                    print("Overlay image could not be read by cv2.")
                    print(overlay_path)
                    return

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # QC preview only: remove large empty top/bottom margins before display.
                # This does NOT change saved overlay files or detection results.
                gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
                non_empty_rows = np.where(np.std(gray, axis=1) > 1.0)[0]

                if non_empty_rows.size > 0:
                    y0 = max(0, int(non_empty_rows.min()) - 10)
                    y1 = min(img.shape[0], int(non_empty_rows.max()) + 11)
                    img = img[y0:y1, :, :]

                h, w = img.shape[:2]

                TARGET_HEIGHT = 220

                scale = TARGET_HEIGHT / max(h, 1)
                new_w = max(1, int(w * scale))

                img_small = cv2.resize(
                    img,
                    (new_w, TARGET_HEIGHT),
                    interpolation=cv2.INTER_AREA
                )

                plt.figure(figsize=(14, 3))
                plt.imshow(img_small)
                plt.axis("off")
                plt.show()

            else:
                print("Overlay image not found.")
                print("Checked default folders:")
                print(os.path.join(OUTPUT_ROOT, f"{PREPROCESS_MODE}_kmeans"))
                print(os.path.join(OUTPUT_ROOT, PREPROCESS_MODE))

    def load_row():
        row = qcv.iloc[state["idx"]]

        manual_val = _clean(row.get("manual", ""))
        manual_dd.value = manual_val if manual_val in manual_dd.options else ""

        quality_val = _clean(row.get("quality_class", "")).lower().strip()
        quality_dd.value = quality_val if quality_val in quality_dd.options else ""

        reason_val = _clean(row.get("reason", "")).lower().strip()
        reason_dd.value = reason_val if reason_val in reason_dd.options else ""

        comment_txt.value = _clean(row.get("comment", ""))

        render()

    def save_row():
        i = state["idx"]

        qcv.loc[i, "manual"] = manual_dd.value
        qcv.loc[i, "quality_class"] = quality_dd.value
        qcv.loc[i, "reason"] = reason_dd.value
        qcv.loc[i, "comment"] = comment_txt.value

        qcv.to_csv(qc_csv_path, index=False)

    def on_next(_):
        save_row()
        state["idx"] = min(state["idx"] + 1, len(qcv) - 1)
        load_row()

    def on_prev(_):
        save_row()
        state["idx"] = max(state["idx"] - 1, 0)
        load_row()

    def on_save(_):
        save_row()
        render()

    def on_stats(_):
        save_row()
        summarize_visual_qc(qc_csv_path)

    next_btn.on_click(on_next)
    prev_btn.on_click(on_prev)
    save_btn.on_click(on_save)
    stats_btn.on_click(on_stats)

    display(
        widgets.VBox([
            out,
            manual_dd,
            quality_dd,
            reason_dd,
            comment_txt,
            widgets.HBox([prev_btn, save_btn, next_btn, stats_btn])
        ])
    )

    load_row()


print("\nTo open the two-slot dropdown QC form, run:")
print("launch_visual_qc_form()")

print("\nAfter QC, run:")
print("summarize_visual_qc()")

try:
    os.startfile(OUTPUT_ROOT)
except Exception:
    print(f"Results saved to: {OUTPUT_ROOT}")


# =====================================================
# STACK ALL RESULT IMAGES AFTER PIPELINE
# =====================================================

if STACK_RESULTS_AT_END:

    for segmentation_method in SEGMENTATION_MODES:

        mode = PREPROCESS_MODE

        output_folder = os.path.join(
            OUTPUT_ROOT,
            f"{mode}_{segmentation_method}"
        )

        stack_result_images(
            folder=output_folder,
            image_type=STACK_IMAGE_TYPE,
            save_png=True,
            open_after_save=STACK_OPEN_AFTER_SAVE
        )


214 files found

PROCESSING ORDER CHECK:
1132_seg0000_3.0m.png -> 1132
1132_seg0001_3.0m.png -> 1132
1132_seg0002_3.0m.png -> 1132
1132_seg0003_3.0m.png -> 1132
1132_seg0004_3.0m.png -> 1132
1132_seg0005_3.0m.png -> 1132
1132_seg0006_3.0m.png -> 1132
1132_seg0007_3.0m.png -> 1132
1132_seg0008_3.0m.png -> 1132
1132_seg0009_3.0m.png -> 1132
1132_seg0010_3.0m.png -> 1132
1132_seg0011_3.0m.png -> 1132
1132_seg0012_3.0m.png -> 1132
1132_seg0013_3.0m.png -> 1132
1132_seg0014_3.0m.png -> 1132
1132_seg0015_3.0m.png -> 1132
1132_seg0016_3.0m.png -> 1132
1132_seg0017_3.0m.png -> 1132
1132_seg0018_3.0m.png -> 1132
1132_seg0019_3.0m.png -> 1132
1132_seg0020_3.0m.png -> 1132
1132_seg0021_3.0m.png -> 1132
1132_seg0022_3.0m.png -> 1132
1132_seg0023_3.0m.png -> 1132
1132_seg0024_3.0m.png -> 1132
Applied image regime preset: cave_rich
ENABLE_PAIR_ISOLATION_GATE = False
MIN_INTERPEAK_VALLEY_DEPTH = 0.45
MAX_INTERPEAK_VALLEY_NOISE = 0.55
MIN_PEAK_TO_BACKGROUND_RATIO = 1.3
BROAD_PLATEAU_MAX_SEPARATION = 1

C:\Users\j091605\AppData\Local\Temp\ipykernel_33920\1253239688.py:2496: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask_clean = remove_small_objects(mask_raw.astype(bool), min_size=MIN_OBJECT_SIZE)
C:\Users\j091605\AppData\Local\Temp\ipykernel_33920\1253239688.py:2497: FutureWarning: Parameter `area_threshold` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_holes`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller

PRE-AZIMUTH VERTICAL LINE SUPPRESSION | removed_pixels=1318, kernel_h=189
Candidates found: 1
NO VALID BREAKOUT CANDIDATE PAIR FOUND

kmeans | IMAGE 2/214
1132_seg0001_3.0m.png
ROI height=2358 px | window height=2358 px | step=1500 px


C:\Users\j091605\AppData\Local\Temp\ipykernel_33920\1253239688.py:2496: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask_clean = remove_small_objects(mask_raw.astype(bool), min_size=MIN_OBJECT_SIZE)
C:\Users\j091605\AppData\Local\Temp\ipykernel_33920\1253239688.py:2497: FutureWarning: Parameter `area_threshold` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_holes`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller

PRE-AZIMUTH VERTICAL LINE SUPPRESSION | removed_pixels=4716, kernel_h=189
Candidates found: 0
NO VALID BREAKOUT CANDIDATE PAIR FOUND

kmeans | IMAGE 3/214
1132_seg0002_3.0m.png
ROI height=2358 px | window height=2358 px | step=1500 px


...

PRE-AZIMUTH VERTICAL LINE SUPPRESSION | removed_pixels=5018, kernel_h=189
Candidates found: 1
NO VALID BREAKOUT CANDIDATE PAIR FOUND

DONE
Automatic detection QC table saved to:
C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\_automatic_qc_tables\automatic_breakout_detection_qc.csv
Automatic detection summary saved to:
C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\_automatic_qc_tables\automatic_breakout_detection_summary.csv

Automatic summary:


,mode,segmentation_method,total_images,code_breakout_detected_count,pair_candidate_found_count,mean_final_mask_pixels,mean_candidate_count,code_breakout_detected_percent,pair_candidate_found_percent
0,bilateral_only,kmeans,214,17,23,12226.626168,2.476636,7.9,10.7



Visual QC table created:
C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\_automatic_qc_tables\visual_qc_kmeans_only_template.csv


,image_file,segmentation_method,bilateral_only_code_detected,pair_candidate_found,pair_quality_passed,candidate_count,final_mask_pixels,final_mask_area_fraction,azimuth_1_deg,azimuth_2_deg,...,peak_bg_ratio,pair_isolation,valley_width_frac,plateau_mean_ratio,weak_effective_peak,status_message,manual,quality_class,reason,comment
0,1132_seg0000_3.0m.png,kmeans,False,False,False,1,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
1,1132_seg0001_3.0m.png,kmeans,False,False,False,0,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
2,1132_seg0002_3.0m.png,kmeans,False,False,False,3,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
3,1132_seg0003_3.0m.png,kmeans,False,False,False,0,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
4,1132_seg0004_3.0m.png,kmeans,False,False,False,3,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,1132_seg0209_3.0m.png,kmeans,False,False,False,3,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
210,1132_seg0210_3.0m.png,kmeans,False,False,False,3,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
211,1132_seg0211_3.0m.png,kmeans,False,False,False,2,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,
212,1132_seg0212_3.0m.png,kmeans,False,False,False,2,0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,no_valid_pair_found,,,,



To open the two-slot dropdown QC form, run:
launch_visual_qc_form()

After QC, run:
summarize_visual_qc()

STACKING | type=overlay | found=268 images
STACKING | total height 3532284px is large; saving in chunks
STACKING | saved chunk OK: True C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\bilateral_only_kmeans\STACKED_overlay_part01.png
STACKING | saved chunk OK: True C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\bilateral_only_kmeans\STACKED_overlay_part02.png
STACKING | saved chunk OK: True C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\bilateral_only_kmeans\STACKED_overlay_part03.png
STACKING | saved chunk OK: True C:\Users\j091605\Desktop\1132\1132\breakout_preannotator_v20_full_image_folder_mode_stacked_qc\bilateral_only_kmeans\STACKED_overlay_part04.png
STACKING | saved chunk OK: True C:\Users\j091605\Desktop\1132\1132\breakout_preannot

In [ ]:
launch_visual_qc_form()

,mode,TP_presence,FP_presence,FN_presence,TN_presence,presence_precision,presence_recall,presence_accuracy,presence_f1,good,fair,poor,usable_annotations,usable_percent,n_images
0,bilateral_only_kmeans,14,3,20,177,0.824,0.412,0.893,0.549,188,10,16,198,92.5,214


,quality_class,count
0,good,188
1,poor,16
2,fair,10


,reason,count
0,missed_breakout,18
1,none,3
2,undersegmented,3
3,artifact_leakage,2
4,oversegmented,2
5,same_plateau_multipeak_false_positive,1
